# Curiosity wheel anomaly detection

Notebook Kaggle autosufficiente. Individua il dataset, configura il preprocessing per modello, prepara i DataLoader e definisce il contratto comune di modelli, metriche e artefatti.

In [ ]:
!pip install -q google-api-python-client google-auth tqdm "datasets[vision]>=4.0,<5"


In [ ]:
from __future__ import annotations

import copy
import csv
import hashlib
import json
import math
import os
import random
import shutil
import tarfile
import urllib.request
from collections import Counter, deque
from collections.abc import Iterable, Mapping
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence

import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, IterableDataset
from torchvision.models import get_model, get_model_weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from torchvision.transforms.functional import pil_to_tensor

from torchvision.transforms.functional import gaussian_blur
from tqdm import tqdm

In [ ]:
# Configuration
KAGGLE_INPUT = Path("/kaggle/input")
BATCH_SIZE = 4
NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# PatchCore presets: Amazon-compatible reference and project-efficient light.
MODEL_NAME = "efficientad_s"  # patchcore | efficientad_s | supersimplenet | tinyglass
PATCHCORE_PRESET = "light"  # ["light", "reference"]
PATCHCORE_PRESETS = {
    "light": {
        "backbone": "resnet18", "input_size": (384, 512),
        "resize": (384, 512), "resize_shorter_side": None, "center_crop": None,
        "num_neighbors": 9, "pretrain_embed_dimension": 384,
        "target_embed_dimension": 384, "projection_dim": 64, "sampling_seed": SEED,
    },
    "reference": {
        "backbone": "wide_resnet50_2", "input_size": (224, 224),
        "resize": None, "resize_shorter_side": 256, "center_crop": (224, 224),
        "num_neighbors": 1, "pretrain_embed_dimension": 1024,
        "target_embed_dimension": 1024, "projection_dim": 128, "sampling_seed": 0,
    },
}
if PATCHCORE_PRESET not in PATCHCORE_PRESETS:
    raise ValueError("PATCHCORE_PRESET must be one of: light, reference")
PATCHCORE_CONFIG = PATCHCORE_PRESETS[PATCHCORE_PRESET]
PATCHCORE_BACKBONE = PATCHCORE_CONFIG["backbone"]
PATCHCORE_PRETRAINED = True
PATCHCORE_LAYERS = ("layer2", "layer3")
PATCHCORE_CORESET_SAMPLING_RATIO = 0.1
PATCHCORE_NUM_NEIGHBORS = PATCHCORE_CONFIG["num_neighbors"]
PATCHCORE_PATCH_SIZE = 3
PATCHCORE_PATCH_STRIDE = 1
PATCHCORE_PRETRAIN_EMBED_DIMENSION = PATCHCORE_CONFIG["pretrain_embed_dimension"]
PATCHCORE_TARGET_EMBED_DIMENSION = PATCHCORE_CONFIG["target_embed_dimension"]
PATCHCORE_MAX_PATCHES_PER_IMAGE = 128
PATCHCORE_MAX_TRAINING_EMBEDDINGS = 50000
PATCHCORE_MAX_MEMORY_BANK_SIZE = 2048
PATCHCORE_PROJECTION_DIM = PATCHCORE_CONFIG["projection_dim"]
PATCHCORE_SAMPLING_SEED = PATCHCORE_CONFIG["sampling_seed"]
PATCHCORE_CALIBRATION_QUANTILE = 0.99
PATCHCORE_CALIBRATION_BATCHES = 20
PATCHCORE_DISTANCE_QUERY_CHUNK_SIZE = 512
PATCHCORE_DISTANCE_BANK_CHUNK_SIZE = 2048
PATCHCORE_GAUSSIAN_SIGMA = 4.0
IMAGE_SIZE = PATCHCORE_CONFIG["input_size"]
PREPROCESS_RESIZE = PATCHCORE_CONFIG["resize"]
PREPROCESS_RESIZE_SHORTER_SIDE = PATCHCORE_CONFIG["resize_shorter_side"]
PREPROCESS_CENTER_CROP = PATCHCORE_CONFIG["center_crop"]
MODEL_RUN_NAME = (
    f"patchcore_{PATCHCORE_PRESET}_{PATCHCORE_BACKBONE}_"
    f"{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}"
)

# PatchCore fits a memory bank without gradient-based optimization.
OPTIMIZER_NAME = "none"
SCHEDULER_NAME = "none"

# Model-specific direct-resize baseline selected above.
NORMALIZE_MEAN = (0.485, 0.456, 0.406)
NORMALIZE_STD = (0.229, 0.224, 0.225)
TRAIN_AUGMENTATIONS_ENABLED = False
TRAIN_BRIGHTNESS = 0.08
TRAIN_CONTRAST = 0.08
TRAIN_GAMMA = 0.08
TRAIN_SATURATION = 0.06
TRAIN_SENSOR_NOISE = 0.004
TRAIN_GAUSSIAN_NOISE = 0.004
TRAIN_GAUSSIAN_BLUR = (0.1, 0.5)

# Full, repository-aligned model configurations.
FIXED_TRAINING_DURATION = True
EFFICIENTAD_TEACHER_URL = (
    "https://raw.githubusercontent.com/nelson1425/EfficientAD/"
    "fcab5146f84a/models/teacher_small.pth"
)
EFFICIENTAD_PENALTY_DATASET_ID = "ILSVRC/imagenet-1k"
EFFICIENTAD_PENALTY_DATASET_REVISION = "49e2ee26f3810fb5a7536bbf732a7b07389a47b5a"
EFFICIENTAD_PENALTY_CACHE_DIR = Path("/kaggle/working/imagenet_penalty_cache")
EFFICIENTAD_PENALTY_TOTAL_SHARDS = 294
EFFICIENTAD_PENALTY_CACHE_NUM_SHARDS = 18
EFFICIENTAD_PENALTY_SHUFFLE_BUFFER = 10_000
DTD_URL = "https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz"
MODEL_CONFIGS = {
    "efficientad_s": {
        "teacher_weights_path": None, "require_teacher_weights": True,
        "channels": 384, "max_steps": 70_000,
        "learning_rate": 1e-4, "weight_decay": 1e-5,
        "hard_quantile": 0.999, "checkpoint_interval": 1_000,
        "batch_size": 1,
    },
    "supersimplenet": {
        "backbone": "wide_resnet50_2", "pretrained": True,
        "weights_name": "IMAGENET1K_V1", "layers": ("layer2", "layer3"),
        "input_size": (256, 256), "patch_size": 3, "epochs": 300,
        "noise_std": 0.015, "perlin_threshold": 0.2,
        "adaptor_learning_rate": 1e-4,
        "segmentation_learning_rate": 2e-4,
        "decision_learning_rate": 2e-4, "scheduler_gamma": 0.4,
        "stop_grad": True, "adapt_classification_features": False,
        "gradient_clip": False, "margin": 0.5, "gaussian_sigma": 4.0,
        "fixed_training_duration": FIXED_TRAINING_DURATION,
        "validation_interval": 4, "validation_batches": 64,
        "max_samples_per_epoch": None, "batch_size": 32,
    },
    "tinyglass": {
        "pretrained": True, "weights_name": "IMAGENET1K_V1",
        "input_size": (256, 256), "patch_size": 3, "epochs": 640,
        "learning_rate": 1e-4, "weight_decay": 1e-2,
        "noise_std": 0.015, "radius_quantile": 0.75,
        "hard_mining_quantile": 0.5, "gas_steps": 20,
        "gas_step_size": 0.001, "hypersphere_projection": True,
        "max_samples_per_epoch": 392,
        "texture_root": None, "require_texture_dataset": True,
        "blend_mean": 0.5, "blend_std": 0.1, "gaussian_sigma": 4.0,
        "fixed_training_duration": FIXED_TRAINING_DURATION,
        "validation_interval": 1, "validation_batches": 64,
        "batch_size": 8,
    },
}
SELECTED_MODEL_CONFIG = MODEL_CONFIGS.get(MODEL_NAME, {})
if MODEL_NAME != "patchcore":
    IMAGE_SIZE = (256, 256)
    PREPROCESS_RESIZE = IMAGE_SIZE
    PREPROCESS_RESIZE_SHORTER_SIDE = None
    PREPROCESS_CENTER_CROP = None
    MODEL_RUN_NAME = f"{MODEL_NAME}_full"
    BATCH_SIZE = SELECTED_MODEL_CONFIG["batch_size"]
    if MODEL_NAME == "efficientad_s":
        NORMALIZE_MEAN = None
        NORMALIZE_STD = None

# Threshold-free evaluation metrics.
METRIC_NAMES = (
    "image_auroc",
    "image_average_precision",
    "pixel_auroc",
    "pixel_average_precision",
)
METRIC_HISTOGRAM_BINS = 2048
RESTRICT_PIXELS_TO_TARGET_MASK = False

# Extended diagnostics and failure analysis.
DIAGNOSTICS_ENABLED = True
DIAGNOSTICS_GROUP_FIELDS = ("severity", "camera_pose", "lighting", "wear")
DIAGNOSTICS_NUM_EXTREME_EXAMPLES = 6
DIAGNOSTICS_PRO_BINS = 256
DIAGNOSTICS_PRO_MAX_FPR = 0.30

# Qualitative prediction visualization.
VISUALIZATION_ENABLED = True
VISUALIZATION_NUM_CLEAN = 3
VISUALIZATION_NUM_ANOMALOUS = 3
VISUALIZATION_COLORMAP = "magma"
VISUALIZATION_OVERLAY_ALPHA = 0.55
VISUALIZATION_DPI = 150

# Training resume and optional Google Drive persistence through Kaggle Secrets.
RESUME = False
RESUME_RUN_DIR = None
DRIVE_UPLOAD_ENABLED = False
DRIVE_PARENT_FOLDER_ID = None
DRIVE_UPLOAD_CHECKPOINTS = True
RUN_ID = (
    f"{MODEL_RUN_NAME}__seed{SEED}__"
    f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
)
if RESUME and not RESUME_RUN_DIR:
    raise ValueError("RESUME=True requires RESUME_RUN_DIR")
if RESUME:
    OUTPUT_DIR = Path(RESUME_RUN_DIR)
    RUN_ID = OUTPUT_DIR.name
else:
    OUTPUT_DIR = Path("/kaggle/working") / RUN_ID

# Visual preprocessing audit.
AUDIT_ENABLED = True
AUDIT_SPLIT = "train"
AUDIT_INDICES = (0, 1, 2)
AUDIT_VARIANTS = 2

# Expected contract for dataset version v1_10000.
EXPECTED_SPLIT_CONDITION_COUNTS = Counter({
    ("train", "clean"): 7000,
    ("validation", "clean"): 750,
    ("validation", "hole"): 250,
    ("test", "clean"): 1000,
    ("test", "hole"): 1000,
})

## Definitions

The following cells only define reusable classes and functions; they do not start the experiment.

In [ ]:
RGBTriplet = tuple[float, float, float]
ImageSize = tuple[int, int]
SigmaRange = tuple[float, float]

@dataclass(frozen=True)
class PreprocessingConfig:
    """Optional preprocessing and train-time image augmentation settings.

    ``resize`` follows the torchvision convention ``(height, width)``. Every
    scalar augmentation value is disabled when ``None`` and otherwise denotes
    its maximum intensity. Photometric transforms affect RGB only.
    """

    resize: ImageSize | None = None
    resize_shorter_side: int | None = None
    center_crop: ImageSize | None = None
    normalize_mean: RGBTriplet | None = None
    normalize_std: RGBTriplet | None = None
    augmentations_enabled: bool = True
    brightness: float | None = None
    contrast: float | None = None
    gamma: float | None = None
    saturation: float | None = None
    sensor_noise: float | None = None
    gaussian_noise: float | None = None
    gaussian_blur: SigmaRange | None = None

    def __post_init__(self) -> None:
        if self.resize is not None and self.resize_shorter_side is not None:
            raise ValueError("resize and resize_shorter_side are mutually exclusive")
        if self.resize is not None:
            if len(self.resize) != 2 or any(value < 1 for value in self.resize):
                raise ValueError("resize must be a positive (height, width) pair")
        if self.resize_shorter_side is not None and self.resize_shorter_side < 1:
            raise ValueError("resize_shorter_side must be positive")
        if self.center_crop is not None:
            if len(self.center_crop) != 2 or any(value < 1 for value in self.center_crop):
                raise ValueError("center_crop must be a positive (height, width) pair")

        if (self.normalize_mean is None) != (self.normalize_std is None):
            raise ValueError("normalize_mean and normalize_std must be set together")
        if self.normalize_mean is not None:
            if len(self.normalize_mean) != 3 or len(self.normalize_std or ()) != 3:
                raise ValueError("normalization mean and std must contain three RGB values")
            if any(value <= 0 for value in self.normalize_std or ()):
                raise ValueError("normalization std values must be positive")

        for name in (
            "brightness",
            "contrast",
            "gamma",
            "saturation",
            "sensor_noise",
            "gaussian_noise",
        ):
            value = getattr(self, name)
            if value is not None and value < 0:
                raise ValueError(f"{name} must be non-negative or None")

        for name in ("brightness", "contrast", "gamma", "saturation"):
            value = getattr(self, name)
            if value is not None and value >= 1:
                raise ValueError(f"{name} must be smaller than 1")

        if self.gaussian_blur is not None:
            if len(self.gaussian_blur) != 2:
                raise ValueError("gaussian_blur must be a (min_sigma, max_sigma) pair")
            minimum, maximum = self.gaussian_blur
            if minimum <= 0 or maximum < minimum:
                raise ValueError("gaussian_blur requires 0 < min_sigma <= max_sigma")


class WheelPreprocessor:
    """Apply aligned geometric transforms and RGB-only photometric transforms."""

    def __init__(self, config: PreprocessingConfig | None = None) -> None:
        self.config = config or PreprocessingConfig()

    @staticmethod
    def _symmetric_factor(maximum_delta: float) -> float:
        return 1.0 + (2.0 * torch.rand(1).item() - 1.0) * maximum_delta

    @staticmethod
    def _odd_kernel_size(maximum_sigma: float) -> int:
        radius = max(1, round(3.0 * maximum_sigma))
        return 2 * radius + 1

    def __call__(
        self,
        image: torch.Tensor,
        target_mask: torch.Tensor,
        anomaly_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        config = self.config

        if config.resize is not None:
            image = TF.resize(
                image,
                config.resize,
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            target_mask = TF.resize(
                target_mask,
                config.resize,
                interpolation=InterpolationMode.NEAREST,
            )
            anomaly_mask = TF.resize(
                anomaly_mask,
                config.resize,
                interpolation=InterpolationMode.NEAREST,
            )
        elif config.resize_shorter_side is not None:
            image = TF.resize(
                image,
                config.resize_shorter_side,
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            target_mask = TF.resize(
                target_mask,
                config.resize_shorter_side,
                interpolation=InterpolationMode.NEAREST,
            )
            anomaly_mask = TF.resize(
                anomaly_mask,
                config.resize_shorter_side,
                interpolation=InterpolationMode.NEAREST,
            )

        if config.center_crop is not None:
            image = TF.center_crop(image, config.center_crop)
            target_mask = TF.center_crop(target_mask, config.center_crop)
            anomaly_mask = TF.center_crop(anomaly_mask, config.center_crop)

        # Every model-facing split uses one stable numerical contract.  Keeping
        # uint8 for evaluation while train-time augmentation returned float32
        # would either fail in the backbone or change the input scale by 255x.
        image = TF.convert_image_dtype(image, torch.float32)

        if config.augmentations_enabled:
            if config.brightness is not None:
                image = TF.adjust_brightness(image, self._symmetric_factor(config.brightness))
            if config.contrast is not None:
                image = TF.adjust_contrast(image, self._symmetric_factor(config.contrast))
            if config.gamma is not None:
                image = TF.adjust_gamma(image, self._symmetric_factor(config.gamma))
            if config.saturation is not None:
                image = TF.adjust_saturation(image, self._symmetric_factor(config.saturation))

            if config.gaussian_blur is not None:
                minimum, maximum = config.gaussian_blur
                sigma = minimum + torch.rand(1).item() * (maximum - minimum)
                kernel_size = self._odd_kernel_size(maximum)
                image = TF.gaussian_blur(image, kernel_size, sigma)

            if config.sensor_noise is not None and config.sensor_noise > 0:
                signal_scale = image.clamp(0.0, 1.0).sqrt()
                image = image + torch.randn_like(image) * signal_scale * config.sensor_noise
            if config.gaussian_noise is not None and config.gaussian_noise > 0:
                image = image + torch.randn_like(image) * config.gaussian_noise

        image = image.clamp(0.0, 1.0)
        if config.normalize_mean is not None:
            image = TF.normalize(
                image,
                mean=config.normalize_mean,
                std=config.normalize_std,
            )

        return image, target_mask, anomaly_mask

    def image_for_display(self, image: torch.Tensor) -> torch.Tensor:
        """Return a float RGB image in [0, 1], undoing normalization if needed."""
        if image.dtype == torch.uint8:
            return TF.convert_image_dtype(image, torch.float32)

        result = image.detach().clone()
        if self.config.normalize_mean is not None:
            mean = result.new_tensor(self.config.normalize_mean).view(3, 1, 1)
            std = result.new_tensor(self.config.normalize_std).view(3, 1, 1)
            result = result * std + mean
        return result.clamp(0.0, 1.0)


In [ ]:
# Kaggle exposes attached datasets as extracted directories.
def find_dataset_root(input_root: Path = KAGGLE_INPUT) -> Path:
    candidates = [
        manifest.parent
        for manifest in input_root.rglob("samples.csv")
        if (manifest.parent / "images").is_dir()
        and (manifest.parent / "masks").is_dir()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"No valid dataset found under {input_root}: "
            "samples.csv, images/, and masks/ are required."
        )
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple dataset candidates found: {candidates}")
    return candidates[0]

In [ ]:
# Self-contained copy of the loader used by the terminal script.
REQUIRED_COLUMNS = {
    "image_id",
    "split",
    "condition",
    "image_path",
    "target_mask_path",
    "anomaly_mask_path",
    "pair_id",
}
VALID_SPLITS = ("train", "validation", "test")
VALID_CONDITIONS = {"clean", "hole"}
ARTIFACT_PATH_FIELDS = ("image_path", "target_mask_path", "anomaly_mask_path")


class CuriosityWheelDataset(Dataset[dict[str, Any]]):
    """Load one split of the extracted Curiosity wheel dataset."""

    def __init__(
        self,
        root: str | Path,
        split: str,
        preprocessing: WheelPreprocessor | None = None,
    ) -> None:
        self.root = Path(root).expanduser().resolve()
        self.split = split
        self.preprocessing = preprocessing or WheelPreprocessor()

        if split not in VALID_SPLITS:
            raise ValueError(f"Unsupported split {split!r}; expected one of {VALID_SPLITS}")
        if not self.root.is_dir():
            raise NotADirectoryError(f"Dataset root is not a directory: {self.root}")

        manifest_path = self.root / "samples.csv"
        if not manifest_path.is_file():
            raise FileNotFoundError(f"Dataset manifest is missing: {manifest_path}")

        with manifest_path.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            missing = REQUIRED_COLUMNS - set(reader.fieldnames or [])
            if missing:
                raise ValueError(f"samples.csv is missing required columns: {sorted(missing)}")
            rows = list(reader)

        self._validate_manifest_rows(rows)
        self.rows = [row for row in rows if row["split"] == split]

        if not self.rows:
            raise ValueError(f"No samples found for split {split!r}")

        for row in self.rows:
            for field in ARTIFACT_PATH_FIELDS:
                relative_path = row[field]
                if relative_path and not (self.root / relative_path).is_file():
                    raise FileNotFoundError(
                        f"Dataset artifact is missing: {self.root / relative_path}"
                    )

    @classmethod
    def _validate_manifest_rows(cls, rows: list[dict[str, str]]) -> None:
        if not rows:
            raise ValueError("samples.csv contains no samples")
        image_ids: set[str] = set()
        pairs: dict[str, list[dict[str, str]]] = {}
        for row in rows:
            image_id = row["image_id"]
            if not image_id:
                raise ValueError("samples.csv contains an empty image_id")
            if image_id in image_ids:
                raise ValueError(f"Duplicate image_id in samples.csv: {image_id}")
            image_ids.add(image_id)
            row_split = row["split"]
            if row_split not in VALID_SPLITS:
                raise ValueError(
                    f"Unsupported split {row_split!r} in sample {image_id}; "
                    f"expected one of {VALID_SPLITS}"
                )
            condition = row["condition"]
            if condition not in VALID_CONDITIONS:
                raise ValueError(f"Unsupported condition {condition!r} in sample {image_id}")
            if not row["image_path"] or not row["target_mask_path"]:
                raise ValueError(f"Sample has incomplete artifact paths: {image_id}")
            if condition == "hole" and not row["anomaly_mask_path"]:
                raise ValueError(f"Hole sample has no anomaly mask: {image_id}")
            if condition == "clean" and row["anomaly_mask_path"]:
                raise ValueError(f"Clean sample unexpectedly has an anomaly mask: {image_id}")
            for field in ARTIFACT_PATH_FIELDS:
                if row[field]:
                    cls._validate_relative_path(row[field])
            pair_id = row["pair_id"]
            if condition == "hole" and not pair_id:
                raise ValueError(f"Hole sample has no pair_id: {image_id}")
            if pair_id:
                pairs.setdefault(pair_id, []).append(row)

        for pair_id, pair_rows in pairs.items():
            conditions = {row["condition"] for row in pair_rows}
            splits = {row["split"] for row in pair_rows}
            if len(pair_rows) != 2 or conditions != VALID_CONDITIONS:
                raise ValueError(
                    f"Pair {pair_id!r} must contain exactly one clean and one hole sample"
                )
            if len(splits) != 1:
                raise ValueError(f"Pair {pair_id!r} crosses dataset splits: {sorted(splits)}")

    @staticmethod
    def _validate_relative_path(value: str) -> None:
        path = Path(value)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError(f"Invalid dataset-relative path: {value!r}")

    def _load_image(self, relative_path: str, mode: str) -> Image.Image:
        path = self.root / relative_path
        if not path.is_file():
            raise FileNotFoundError(f"Dataset artifact is missing: {path}")
        with Image.open(path) as image:
            return image.convert(mode)

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.rows[index]
        image = pil_to_tensor(self._load_image(row["image_path"], "RGB"))
        target_mask = pil_to_tensor(self._load_image(row["target_mask_path"], "L"))

        if image.shape[1:] != target_mask.shape[1:]:
            raise ValueError(f"Image/target mask size mismatch for {row['image_id']}")

        anomaly_path = row["anomaly_mask_path"]
        if anomaly_path:
            anomaly_mask = pil_to_tensor(self._load_image(anomaly_path, "L"))
            if anomaly_mask.shape != target_mask.shape:
                raise ValueError(f"Target/anomaly mask size mismatch for {row['image_id']}")
        else:
            # Clean samples use an empty mask so every batch has one stable schema.
            anomaly_mask = torch.zeros_like(target_mask)

        image, target_mask, anomaly_mask = self.preprocessing(
            image, target_mask, anomaly_mask
        )

        return {
            "image": image,
            "target_mask": target_mask,
            "anomaly_mask": anomaly_mask,
            "label": torch.tensor(row["condition"] == "hole", dtype=torch.long),
            "has_anomaly_mask": bool(anomaly_path),
            "metadata": {
                key: value
                for key, value in row.items()
                if key not in {"image_path", "target_mask_path", "anomaly_mask_path"}
            },
        }

In [ ]:
def build_dataloader(
    root: str | Path,
    split: str,
    *,
    batch_size: int = 4,
    num_workers: int = 0,
    pin_memory: bool = False,
    seed: int = 42,
    preprocessing: WheelPreprocessor | None = None,
) -> DataLoader:
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")
    if num_workers < 0:
        raise ValueError("num_workers cannot be negative")

    dataset = CuriosityWheelDataset(root, split, preprocessing=preprocessing)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=split == "train",
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
        generator=torch.Generator().manual_seed(seed),
    )


def build_dataloaders(
    root: str | Path,
    *,
    batch_size: int = 4,
    num_workers: int = 0,
    pin_memory: bool = False,
    seed: int = 42,
    train_preprocessing: WheelPreprocessor | None = None,
    evaluation_preprocessing: WheelPreprocessor | None = None,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    def make_loader(split: str) -> DataLoader:
        return build_dataloader(
            root,
            split,
            batch_size=batch_size,
            num_workers=num_workers,
            pin_memory=pin_memory,
            seed=seed,
            preprocessing=train_preprocessing if split == "train" else evaluation_preprocessing,
        )

    return (
        make_loader("train"),
        make_loader("validation"),
        make_loader("test"),
    )

def select_imagenet_penalty_shards(*, total_shards, num_shards, seed):
    if total_shards < 1:
        raise ValueError("total_shards must be at least 1")
    if not 1 <= num_shards <= total_shards:
        raise ValueError("num_shards must be between 1 and total_shards")
    indices = sorted(random.Random(seed).sample(range(total_shards), num_shards))
    return tuple(
        f"data/train-{index:05d}-of-{total_shards:05d}.parquet"
        for index in indices
    )


def prepare_imagenet_penalty_cache(token):
    if not token:
        raise RuntimeError(
            "Add an HF_TOKEN Kaggle secret after accepting ILSVRC/imagenet-1k access"
        )
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
    from huggingface_hub import hf_hub_download
    import pyarrow.parquet as parquet

    cache_dir = EFFICIENTAD_PENALTY_CACHE_DIR.resolve()
    cache_dir.mkdir(parents=True, exist_ok=True)
    filenames = select_imagenet_penalty_shards(
        total_shards=EFFICIENTAD_PENALTY_TOTAL_SHARDS,
        num_shards=EFFICIENTAD_PENALTY_CACHE_NUM_SHARDS,
        seed=SEED,
    )
    free_gib = shutil.disk_usage(cache_dir).free / 2**30
    print(
        f"[ImageNet cache] Preparing {len(filenames)} deterministic shards in "
        f"{cache_dir}; free disk={free_gib:.1f} GiB. Existing files are reused.",
        flush=True,
    )

    local_files = []
    for position, filename in enumerate(filenames, start=1):
        local_path = cache_dir / filename
        if local_path.is_file() and local_path.stat().st_size > 0:
            print(
                f"[ImageNet cache] [{position}/{len(filenames)}] Reusing "
                f"{local_path.name} ({local_path.stat().st_size / 2**20:.1f} MiB).",
                flush=True,
            )
        else:
            print(
                f"[ImageNet cache] [{position}/{len(filenames)}] Downloading "
                f"{filename}. Rerun the cell after an interruption to resume.",
                flush=True,
            )
            try:
                downloaded = hf_hub_download(
                    repo_id=EFFICIENTAD_PENALTY_DATASET_ID,
                    filename=filename,
                    repo_type="dataset",
                    revision=EFFICIENTAD_PENALTY_DATASET_REVISION,
                    token=token,
                    local_dir=cache_dir,
                )
            except Exception as error:
                raise RuntimeError(
                    f"Failed to cache {filename}. Rerun this cell; completed "
                    "shards will be reused."
                ) from error
            local_path = Path(downloaded).resolve()
        local_files.append(local_path)

    total_rows = 0
    total_bytes = 0
    for local_path in local_files:
        try:
            total_rows += parquet.ParquetFile(local_path).metadata.num_rows
        except Exception as error:
            raise RuntimeError(
                f"Unreadable cached Parquet shard: {local_path}. Delete only "
                "this file and rerun the cell."
            ) from error
        total_bytes += local_path.stat().st_size
    required_samples = MODEL_CONFIGS["efficientad_s"]["max_steps"]
    if total_rows < required_samples:
        raise RuntimeError(
            f"Cached shards contain {total_rows:,} samples but EfficientAD needs "
            f"{required_samples:,}. Increase EFFICIENTAD_PENALTY_CACHE_NUM_SHARDS."
        )
    print(
        f"[ImageNet cache] Ready: {len(local_files)} shards, {total_rows:,} "
        f"samples, {total_bytes / 2**30:.2f} GiB. fit() will read local files only.",
        flush=True,
    )
    return tuple(local_files)


class ImageNetPenaltyDataset(IterableDataset):
    """Read cached ImageNet-1k shards with EfficientAD's penalty transform."""

    def __init__(self, *, local_files, seed=42, shuffle_buffer_size=10_000):
        super().__init__()
        if shuffle_buffer_size < 1:
            raise ValueError("shuffle_buffer_size must be at least 1")
        local_files = tuple(str(Path(path).resolve()) for path in local_files)
        if not local_files:
            raise ValueError("local_files cannot be empty")
        missing = [path for path in local_files if not Path(path).is_file()]
        if missing:
            raise FileNotFoundError(f"Missing cached ImageNet shard: {missing[0]}")
        from datasets import load_dataset
        print(
            f"[ImageNet penalty] Opening {len(local_files)} local Parquet shards; "
            f"shuffle_buffer={shuffle_buffer_size:,}.",
            flush=True,
        )
        stream = load_dataset(
            "parquet",
            data_files={"train": list(local_files)},
            split="train",
            streaming=True,
        )
        self._stream = stream.shuffle(seed=seed, buffer_size=shuffle_buffer_size)
        print(
            "[ImageNet penalty] Local stream configured; no ImageNet network reads "
            "will occur during fit().",
            flush=True,
        )

    @staticmethod
    def _transform(image):
        image = image.convert("RGB")
        image = TF.resize(
            image, [512, 512], interpolation=InterpolationMode.BILINEAR, antialias=True
        )
        if random.random() < 0.3:
            image = TF.rgb_to_grayscale(image, num_output_channels=3)
        image = TF.center_crop(image, [256, 256])
        return TF.pil_to_tensor(image).float().div_(255.0)

    def __iter__(self):
        print(
            "[ImageNet penalty] Iterator started; requesting the first local sample...",
            flush=True,
        )
        for index, sample in enumerate(self._stream, start=1):
            if index == 1:
                print(
                    "[ImageNet penalty] First local sample received; applying the "
                    "EfficientAD penalty transform.",
                    flush=True,
                )
            yield {"image": self._transform(sample["image"])}
            if index % 1_000 == 0:
                print(
                    f"[ImageNet penalty] Local stream yielded {index:,} samples.",
                    flush=True,
                )

    def state_dict(self):
        return self._stream.state_dict()

    def load_state_dict(self, state_dict):
        self._stream.load_state_dict(state_dict)


def build_imagenet_penalty_loader(token):
    local_files = prepare_imagenet_penalty_cache(token)
    print(
        f"[ImageNet penalty] Building local DataLoader with num_workers=0 and "
        f"pin_memory={PIN_MEMORY}.",
        flush=True,
    )
    return DataLoader(
        ImageNetPenaltyDataset(
            local_files=local_files,
            seed=SEED,
            shuffle_buffer_size=EFFICIENTAD_PENALTY_SHUFFLE_BUFFER,
        ),
        batch_size=1,
        num_workers=0,
        pin_memory=PIN_MEMORY,
    )


In [ ]:
@dataclass(frozen=True)
class AnomalyPrediction:
    anomaly_score: torch.Tensor
    anomaly_map: torch.Tensor

    def __post_init__(self):
        if self.anomaly_score.ndim != 1:
            raise ValueError("anomaly_score must have shape [B]")
        if self.anomaly_map.ndim != 4 or self.anomaly_map.shape[1] != 1:
            raise ValueError("anomaly_map must have shape [B, 1, H, W]")
        if self.anomaly_score.shape[0] != self.anomaly_map.shape[0]:
            raise ValueError("anomaly_score and anomaly_map batch sizes must match")
        for name, value in (("anomaly_score", self.anomaly_score), ("anomaly_map", self.anomaly_map)):
            if not value.is_floating_point() or not torch.isfinite(value).all():
                raise ValueError(f"{name} must contain finite floating-point values")
            if value.numel() and (value.min() < 0 or value.max() > 1):
                raise ValueError(f"{name} must be normalized to [0, 1]")


class AnomalyDetector(nn.Module):
    @property
    def is_fitted(self):
        raise NotImplementedError

    def fit(self, train_loader: Iterable[Mapping[str, Any]], *, device):
        raise NotImplementedError

    @torch.no_grad()
    def predict(self, images):
        raise NotImplementedError

    def checkpoint_config(self):
        return {}

    def save(self, path, *, metadata=None):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "schema_version": 2,
            "model_class": f"{type(self).__module__}.{type(self).__qualname__}",
            "model_config": self.checkpoint_config(),
            "model_state_dict": self.state_dict(),
            "metadata": dict(metadata or {}),
        }, path)
        return path

    def _prepare_state_dict_for_load(self, state_dict):
        pass

    def load(self, path, *, map_location=None):
        payload = torch.load(Path(path), map_location=map_location)
        if not isinstance(payload, dict) or "model_state_dict" not in payload:
            raise ValueError("Checkpoint does not contain model_state_dict")
        if payload.get("schema_version") != 2:
            raise ValueError(f"Unsupported checkpoint schema {payload.get('schema_version')!r}")
        expected_class = f"{type(self).__module__}.{type(self).__qualname__}"
        if payload.get("model_class") != expected_class:
            raise ValueError(
                f"Checkpoint model class {payload.get('model_class')!r} does not "
                f"match {expected_class!r}"
            )
        saved_config = payload.get("model_config")
        expected_config = self.checkpoint_config()
        if saved_config != expected_config:
            raise ValueError(
                "Checkpoint model configuration does not match the current model: "
                f"saved={saved_config!r}, current={expected_config!r}"
            )
        state_dict = payload["model_state_dict"]
        self._prepare_state_dict_for_load(state_dict)
        self.load_state_dict(state_dict)
        return dict(payload.get("metadata", {}))

In [ ]:
from __future__ import annotations

class PatchCore(AnomalyDetector):
    """Memory-bounded PatchCore anomaly detector with a frozen backbone."""

    def __init__(
        self,
        *,
        backbone: str = "resnet18",
        pretrained: bool = True,
        layers: Sequence[str] = ("layer2", "layer3"),
        coreset_sampling_ratio: float = 0.1,
        num_neighbors: int = 9,
        patch_size: int = 3,
        patch_stride: int = 1,
        pretrain_embed_dimension: int = 384,
        target_embed_dimension: int = 384,
        max_patches_per_image: int = 128,
        max_training_embeddings: int = 50_000,
        max_memory_bank_size: int = 2_048,
        projection_dim: int = 64,
        sampling_seed: int = 42,
        calibration_quantile: float = 0.99,
        calibration_batches: int = 20,
        distance_query_chunk_size: int = 512,
        distance_bank_chunk_size: int = 2_048,
        gaussian_sigma: float = 4.0,
    ) -> None:
        super().__init__()
        if not layers:
            raise ValueError("layers must contain at least one feature node")
        if not 0 < coreset_sampling_ratio <= 1:
            raise ValueError("coreset_sampling_ratio must be in (0, 1]")
        if num_neighbors < 1:
            raise ValueError("num_neighbors must be at least 1")
        if patch_size < 1 or patch_size % 2 == 0:
            raise ValueError("patch_size must be a positive odd integer")
        for name, value in (
            ("max_patches_per_image", max_patches_per_image),
            ("max_training_embeddings", max_training_embeddings),
            ("max_memory_bank_size", max_memory_bank_size),
            ("projection_dim", projection_dim),
            ("patch_stride", patch_stride),
            ("pretrain_embed_dimension", pretrain_embed_dimension),
            ("target_embed_dimension", target_embed_dimension),
            ("calibration_batches", calibration_batches),
            ("distance_query_chunk_size", distance_query_chunk_size),
            ("distance_bank_chunk_size", distance_bank_chunk_size),
        ):
            if value < 1:
                raise ValueError(f"{name} must be at least 1")
        if not 0 < calibration_quantile <= 1:
            raise ValueError("calibration_quantile must be in (0, 1]")
        if gaussian_sigma < 0:
            raise ValueError("gaussian_sigma cannot be negative")

        weights = get_model_weights(backbone).DEFAULT if pretrained else None
        backbone_model = get_model(backbone, weights=weights)
        self.feature_extractor = create_feature_extractor(
            backbone_model,
            return_nodes={layer: layer for layer in layers},
        )
        self.feature_extractor.requires_grad_(False)

        self.backbone = backbone
        self.pretrained = bool(pretrained)
        self.layers = tuple(layers)
        self.coreset_sampling_ratio = float(coreset_sampling_ratio)
        self.num_neighbors = int(num_neighbors)
        self.patch_size = int(patch_size)
        self.patch_stride = int(patch_stride)
        self.pretrain_embed_dimension = int(pretrain_embed_dimension)
        self.target_embed_dimension = int(target_embed_dimension)
        self.max_patches_per_image = int(max_patches_per_image)
        self.max_training_embeddings = int(max_training_embeddings)
        self.max_memory_bank_size = int(max_memory_bank_size)
        self.projection_dim = int(projection_dim)
        self.sampling_seed = int(sampling_seed)
        self.calibration_quantile = float(calibration_quantile)
        self.calibration_batches = int(calibration_batches)
        self.distance_query_chunk_size = int(distance_query_chunk_size)
        self.distance_bank_chunk_size = int(distance_bank_chunk_size)
        self.gaussian_sigma = float(gaussian_sigma)

        self.register_buffer("memory_bank", torch.empty(0, 0), persistent=True)
        self.register_buffer("image_score_scale", torch.tensor(0.0), persistent=True)
        self.register_buffer("anomaly_map_scale", torch.tensor(0.0), persistent=True)
        self.fit_summary: dict[str, int | float | list[int]] = {}
        self.train(False)

    @property
    def is_fitted(self) -> bool:
        return (
            self.memory_bank.ndim == 2
            and self.memory_bank.shape[0] > 0
            and self.image_score_scale.item() > 0
            and self.anomaly_map_scale.item() > 0
        )

    def checkpoint_config(self) -> dict[str, object]:
        return {
            "backbone": self.backbone,
            "layers": list(self.layers),
            "coreset_sampling_ratio": self.coreset_sampling_ratio,
            "num_neighbors": self.num_neighbors,
            "patch_size": self.patch_size,
            "patch_stride": self.patch_stride,
            "pretrain_embed_dimension": self.pretrain_embed_dimension,
            "target_embed_dimension": self.target_embed_dimension,
            "max_patches_per_image": self.max_patches_per_image,
            "max_training_embeddings": self.max_training_embeddings,
            "max_memory_bank_size": self.max_memory_bank_size,
            "projection_dim": self.projection_dim,
            "sampling_seed": self.sampling_seed,
            "calibration_quantile": self.calibration_quantile,
            "calibration_batches": self.calibration_batches,
            "distance_query_chunk_size": self.distance_query_chunk_size,
            "distance_bank_chunk_size": self.distance_bank_chunk_size,
            "gaussian_sigma": self.gaussian_sigma,
        }

    def train(self, mode: bool = True) -> PatchCore:
        """Keep the frozen backbone and its BatchNorm layers in evaluation mode."""
        super().train(False)
        return self

    @staticmethod
    def _validate_images(images: torch.Tensor) -> None:
        if images.ndim != 4 or images.shape[1] != 3:
            raise ValueError(f"Expected RGB images [B, 3, H, W], got {tuple(images.shape)}")
        if not images.is_floating_point() or not torch.isfinite(images).all():
            raise ValueError("images must contain finite floating-point values")

    def _patchify(
        self,
        feature: torch.Tensor,
    ) -> tuple[torch.Tensor, tuple[int, int]]:
        """Extract overlapping local patches without reducing their channels."""
        padding = self.patch_size // 2
        patches = F.unfold(
            feature,
            kernel_size=self.patch_size,
            stride=self.patch_stride,
            padding=padding,
        ).transpose(1, 2)
        height = (feature.shape[-2] + 2 * padding - self.patch_size) // self.patch_stride + 1
        width = (feature.shape[-1] + 2 * padding - self.patch_size) // self.patch_stride + 1
        patches = patches.reshape(
            feature.shape[0],
            height * width,
            feature.shape[1],
            self.patch_size,
            self.patch_size,
        )
        return patches, (height, width)

    @staticmethod
    def _align_patch_grid(
        patches: torch.Tensor,
        source_size: tuple[int, int],
        target_size: tuple[int, int],
    ) -> torch.Tensor:
        """Interpolate patch grids while preserving channel and patch axes."""
        if source_size == target_size:
            return patches
        batch_size, _, channels, patch_height, patch_width = patches.shape
        patches = patches.reshape(
            batch_size,
            *source_size,
            channels,
            patch_height,
            patch_width,
        ).permute(0, 3, 4, 5, 1, 2)
        patches = patches.reshape(-1, 1, *source_size)
        patches = F.interpolate(
            patches,
            size=target_size,
            mode="bilinear",
            align_corners=False,
        )
        return patches.reshape(
            batch_size,
            channels,
            patch_height,
            patch_width,
            *target_size,
        ).permute(0, 4, 5, 1, 2, 3).reshape(
            batch_size,
            target_size[0] * target_size[1],
            channels,
            patch_height,
            patch_width,
        )

    @staticmethod
    def _map_embedding_dimension(
        patches: torch.Tensor,
        output_dimension: int,
    ) -> torch.Tensor:
        """Apply the reference MeanMapper operation to every local patch."""
        batch_size, num_patches = patches.shape[:2]
        flattened = patches.reshape(batch_size * num_patches, 1, -1)
        mapped = F.adaptive_avg_pool1d(flattened, output_dimension)
        return mapped.reshape(batch_size, num_patches, output_dimension)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """Return reference-style locally aggregated embeddings [B, D, h, w]."""
        self._validate_images(images)
        features = self.feature_extractor(images)
        patch_layers = [self._patchify(features[layer]) for layer in self.layers]
        target_size = patch_layers[0][1]
        mapped_layers = []
        for patches, patch_grid in patch_layers:
            patches = self._align_patch_grid(patches, patch_grid, target_size)
            mapped_layers.append(
                self._map_embedding_dimension(
                    patches,
                    self.pretrain_embed_dimension,
                )
            )
        embeddings = torch.cat(mapped_layers, dim=-1)
        embeddings = self._map_embedding_dimension(
            embeddings,
            self.target_embed_dimension,
        )
        return embeddings.transpose(1, 2).reshape(
            images.shape[0],
            self.target_embed_dimension,
            *target_size,
        )

    def _patch_embeddings(
        self,
        images: torch.Tensor,
    ) -> tuple[torch.Tensor, tuple[int, int]]:
        embedding_map = self(images)
        height, width = embedding_map.shape[-2:]
        embeddings = embedding_map.permute(0, 2, 3, 1).flatten(1, 2).contiguous()
        return embeddings, (height, width)

    def _sample_image_patches(
        self,
        embeddings: torch.Tensor,
        generator: torch.Generator,
    ) -> torch.Tensor:
        batch_size, num_patches, embedding_dim = embeddings.shape
        selected_count = min(self.max_patches_per_image, num_patches)
        if selected_count == num_patches:
            return embeddings.reshape(-1, embedding_dim)
        random_keys = torch.rand((batch_size, num_patches), generator=generator)
        indices = random_keys.topk(selected_count, dim=1, largest=False).indices
        indices = indices.to(embeddings.device)
        return embeddings.gather(
            1,
            indices.unsqueeze(-1).expand(-1, -1, embedding_dim),
        ).reshape(-1, embedding_dim)

    @staticmethod
    def _reduce_priority_reservoir(
        embedding_chunks: list[torch.Tensor],
        priority_chunks: list[torch.Tensor],
        maximum_size: int,
    ) -> tuple[list[torch.Tensor], list[torch.Tensor], int]:
        embeddings = torch.cat(embedding_chunks)
        priorities = torch.cat(priority_chunks)
        if embeddings.shape[0] > maximum_size:
            indices = priorities.topk(maximum_size, largest=True, sorted=False).indices
            embeddings = embeddings[indices]
            priorities = priorities[indices]
        return [embeddings], [priorities], embeddings.shape[0]

    def _build_coreset(
        self,
        candidates: torch.Tensor,
        *,
        device: torch.device,
        generator: torch.Generator,
    ) -> torch.Tensor:
        target_size = max(1, math.ceil(candidates.shape[0] * self.coreset_sampling_ratio))
        target_size = min(target_size, self.max_memory_bank_size, candidates.shape[0])
        if target_size == candidates.shape[0]:
            return candidates.contiguous()

        projection_dim = min(self.projection_dim, candidates.shape[1])
        projection = torch.randn(
            candidates.shape[1],
            projection_dim,
            generator=generator,
        ) / math.sqrt(projection_dim)
        projected = candidates.to(device) @ projection.to(device)

        first_index = int(torch.randint(candidates.shape[0], (1,), generator=generator))
        selected = torch.empty(target_size, dtype=torch.long, device=device)
        selected_mask = torch.zeros(candidates.shape[0], dtype=torch.bool, device=device)
        minimum_distances = torch.full(
            (candidates.shape[0],),
            torch.inf,
            dtype=projected.dtype,
            device=device,
        )
        current_index = first_index
        for position in tqdm(range(target_size), desc="PatchCore coreset", leave=False):
            selected[position] = current_index
            selected_mask[current_index] = True
            if position + 1 == target_size:
                break
            center = projected[current_index]
            distances = (projected - center).square().sum(dim=1)
            minimum_distances = torch.minimum(minimum_distances, distances)
            minimum_distances[selected_mask] = -1
            current_index = int(minimum_distances.argmax())
        return candidates[selected.cpu()].contiguous()

    @staticmethod
    def _batch_images(batch: Mapping[str, Any], device: torch.device) -> torch.Tensor:
        if "image" not in batch:
            raise KeyError("Every training batch must contain 'image'")
        labels = batch.get("label")
        if labels is not None and torch.any(torch.as_tensor(labels) != 0):
            raise ValueError("PatchCore.fit accepts only clean training images")
        return batch["image"].to(device, non_blocking=True)

    @torch.no_grad()
    def fit(
        self,
        train_loader: Iterable[Mapping[str, Any]],
        *,
        device: torch.device | str,
    ) -> PatchCore:
        """Build the bounded coreset memory bank and clean-score calibration."""
        device = torch.device(device)
        self.to(device)
        self.train(False)
        self.memory_bank = torch.empty(0, 0, device=device)
        self.image_score_scale.zero_()
        self.anomaly_map_scale.zero_()
        self.fit_summary = {}
        generator = torch.Generator().manual_seed(self.sampling_seed)
        embedding_chunks: list[torch.Tensor] = []
        priority_chunks: list[torch.Tensor] = []
        buffered_count = 0
        num_images = 0
        sampled_embeddings = 0
        patch_grid: tuple[int, int] | None = None

        for batch in tqdm(train_loader, desc="PatchCore feature extraction"):
            images = self._batch_images(batch, device)
            embeddings, patch_grid = self._patch_embeddings(images)
            sampled = self._sample_image_patches(embeddings, generator).cpu()
            priorities = torch.rand(sampled.shape[0], generator=generator)
            embedding_chunks.append(sampled)
            priority_chunks.append(priorities)
            buffered_count += sampled.shape[0]
            sampled_embeddings += sampled.shape[0]
            num_images += images.shape[0]
            if buffered_count >= 2 * self.max_training_embeddings:
                embedding_chunks, priority_chunks, buffered_count = (
                    self._reduce_priority_reservoir(
                        embedding_chunks,
                        priority_chunks,
                        self.max_training_embeddings,
                    )
                )

        if not embedding_chunks or patch_grid is None:
            raise ValueError("train_loader produced no images")
        embedding_chunks, _, candidate_count = self._reduce_priority_reservoir(
            embedding_chunks,
            priority_chunks,
            self.max_training_embeddings,
        )
        candidates = embedding_chunks[0]
        self.memory_bank = self._build_coreset(
            candidates,
            device=device,
            generator=generator,
        ).to(device)

        image_scores: list[torch.Tensor] = []
        patch_scores: list[torch.Tensor] = []
        for batch_index, batch in enumerate(
            tqdm(train_loader, desc="PatchCore calibration", leave=False)
        ):
            if batch_index >= self.calibration_batches:
                break
            images = self._batch_images(batch, device)
            raw_image_scores, raw_patch_scores, _ = self._raw_scores(images)
            image_scores.append(raw_image_scores.cpu())
            patch_scores.append(raw_patch_scores.flatten().cpu())
        if not image_scores:
            raise ValueError("train_loader must be re-iterable for calibration")

        epsilon = torch.finfo(torch.float32).eps
        image_scale = torch.quantile(
            torch.cat(image_scores).float(), self.calibration_quantile
        ).clamp_min(epsilon)
        map_scale = torch.quantile(
            torch.cat(patch_scores).float(), self.calibration_quantile
        ).clamp_min(epsilon)
        self.image_score_scale.copy_(image_scale.to(device))
        self.anomaly_map_scale.copy_(map_scale.to(device))
        self.fit_summary = {
            "num_training_images": num_images,
            "sampled_embeddings": sampled_embeddings,
            "reservoir_embeddings": candidate_count,
            "memory_bank_embeddings": self.memory_bank.shape[0],
            "embedding_dimension": self.memory_bank.shape[1],
            "patch_grid": list(patch_grid),
            "image_score_scale": float(self.image_score_scale),
            "anomaly_map_scale": float(self.anomaly_map_scale),
        }
        return self

    def _nearest_memory(
        self,
        queries: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if self.memory_bank.numel() == 0:
            raise RuntimeError("PatchCore must be fitted before nearest-neighbour search")
        if queries.shape[1] != self.memory_bank.shape[1]:
            raise ValueError("Query and memory-bank embedding dimensions do not match")

        nearest_distances = torch.empty(queries.shape[0], device=queries.device)
        nearest_indices = torch.empty(
            queries.shape[0], dtype=torch.long, device=queries.device
        )
        for query_start in range(0, queries.shape[0], self.distance_query_chunk_size):
            query_end = min(query_start + self.distance_query_chunk_size, queries.shape[0])
            query_chunk = queries[query_start:query_end]
            best_distances = torch.full(
                (query_chunk.shape[0],), torch.inf, device=queries.device
            )
            best_indices = torch.zeros(
                query_chunk.shape[0], dtype=torch.long, device=queries.device
            )
            for bank_start in range(0, self.memory_bank.shape[0], self.distance_bank_chunk_size):
                bank_end = min(
                    bank_start + self.distance_bank_chunk_size,
                    self.memory_bank.shape[0],
                )
                distances = torch.cdist(query_chunk, self.memory_bank[bank_start:bank_end])
                block_distances, block_indices = distances.min(dim=1)
                improved = block_distances < best_distances
                best_distances[improved] = block_distances[improved]
                best_indices[improved] = block_indices[improved] + bank_start
            nearest_distances[query_start:query_end] = best_distances
            nearest_indices[query_start:query_end] = best_indices
        return nearest_distances, nearest_indices

    def _reweighted_image_scores(
        self,
        embeddings: torch.Tensor,
        patch_scores: torch.Tensor,
        nearest_indices: torch.Tensor,
    ) -> torch.Tensor:
        maximum_scores, maximum_locations = patch_scores.max(dim=1)
        if self.num_neighbors == 1 or self.memory_bank.shape[0] == 1:
            return maximum_scores

        scores = []
        num_support = min(self.num_neighbors, self.memory_bank.shape[0])
        for batch_index in range(embeddings.shape[0]):
            patch_index = maximum_locations[batch_index]
            query = embeddings[batch_index, patch_index]
            anchor_index = nearest_indices[batch_index, patch_index]
            anchor = self.memory_bank[anchor_index]
            anchor_distances = torch.linalg.vector_norm(self.memory_bank - anchor, dim=1)
            support_indices = anchor_distances.topk(
                num_support, largest=False
            ).indices
            support_distances = torch.linalg.vector_norm(
                self.memory_bank[support_indices] - query,
                dim=1,
            )
            anchor_position = (support_indices == anchor_index).nonzero(as_tuple=False)[0, 0]
            weight = 1.0 - torch.softmax(support_distances, dim=0)[anchor_position]
            scores.append(maximum_scores[batch_index] * weight)
        return torch.stack(scores)

    def _raw_scores(
        self,
        images: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, tuple[int, int]]:
        embeddings, patch_grid = self._patch_embeddings(images)
        batch_size, num_patches, embedding_dim = embeddings.shape
        distances, nearest_indices = self._nearest_memory(
            embeddings.reshape(-1, embedding_dim)
        )
        patch_scores = distances.reshape(batch_size, num_patches)
        nearest_indices = nearest_indices.reshape(batch_size, num_patches)
        image_scores = self._reweighted_image_scores(
            embeddings, patch_scores, nearest_indices
        )
        return image_scores, patch_scores, patch_grid

    @staticmethod
    def _normalize(raw_scores: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
        epsilon = torch.finfo(raw_scores.dtype).eps
        return raw_scores / (raw_scores + scale.clamp_min(epsilon))

    @torch.no_grad()
    def predict_with_raw(
        self,
        images: torch.Tensor,
    ) -> tuple[AnomalyPrediction, torch.Tensor, torch.Tensor]:
        if not self.is_fitted:
            raise RuntimeError("PatchCore must be fitted before predict")
        input_size = images.shape[-2:]
        raw_image_scores, raw_patch_scores, patch_grid = self._raw_scores(images)
        raw_map = raw_patch_scores.reshape(images.shape[0], 1, *patch_grid)
        raw_map = F.interpolate(
            raw_map,
            size=input_size,
            mode="bilinear",
            align_corners=False,
        )
        if self.gaussian_sigma > 0:
            kernel_size = 2 * math.ceil(3 * self.gaussian_sigma) + 1
            raw_map = gaussian_blur(
                raw_map,
                [kernel_size, kernel_size],
                [self.gaussian_sigma, self.gaussian_sigma],
            )
        prediction = AnomalyPrediction(
            anomaly_score=self._normalize(raw_image_scores, self.image_score_scale),
            anomaly_map=self._normalize(raw_map, self.anomaly_map_scale),
        )
        return prediction, raw_image_scores, raw_map

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        prediction, _, _ = self.predict_with_raw(images)
        return prediction

    def _prepare_state_dict_for_load(self, state_dict: Mapping[str, torch.Tensor]) -> None:
        memory_bank = state_dict.get("memory_bank")
        if memory_bank is not None and memory_bank.shape != self.memory_bank.shape:
            self.memory_bank = torch.empty_like(memory_bank)


import copy
import hashlib
import math
from collections.abc import Iterable, Mapping
from pathlib import Path
from typing import Any

import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torchvision.models import get_model
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms.functional import gaussian_blur, pil_to_tensor



def _clean_images(batch: Mapping[str, Any], device: torch.device) -> torch.Tensor:
    if "image" not in batch:
        raise KeyError("Every training batch must contain 'image'")
    if torch.any(torch.as_tensor(batch.get("label", 0)) != 0):
        raise ValueError("fit accepts only clean training images")
    return batch["image"].to(device, non_blocking=True)


def _image_auc(scores: list[float], labels: list[int]) -> float:
    positives = [score for score, label in zip(scores, labels) if label == 1]
    negatives = [score for score, label in zip(scores, labels) if label == 0]
    if not positives or not negatives:
        return float("nan")
    wins = sum(p > n for p in positives for n in negatives)
    ties = sum(p == n for p in positives for n in negatives)
    return (wins + 0.5 * ties) / (len(positives) * len(negatives))


@torch.no_grad()
def _validation_auc(model, loader, device: torch.device, max_batches: int) -> float:
    if loader is None:
        return float("nan")
    model.eval()
    scores: list[float] = []
    labels: list[int] = []
    for index, batch in enumerate(loader):
        if index >= max_batches:
            break
        prediction = model.predict(batch["image"].to(device, non_blocking=True))
        scores.extend(prediction.anomaly_score.cpu().tolist())
        labels.extend(torch.as_tensor(batch["label"]).int().tolist())
    return _image_auc(scores, labels)


class PDNSmall(nn.Module):
    """EfficientAD PDN-S, including the paper's internal ImageNet normalization."""

    def __init__(self, output_channels: int = 384) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(3, 128, kernel_size=4, padding=3)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1)
        self.conv2 = nn.Conv2d(128, 256, kernel_size=4, padding=3)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1)
        self.conv3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(256, output_channels, kernel_size=4)

    @staticmethod
    def normalize(images: torch.Tensor) -> torch.Tensor:
        mean = images.new_tensor((0.485, 0.456, 0.406)).view(1, 3, 1, 1)
        std = images.new_tensor((0.229, 0.224, 0.225)).view(1, 3, 1, 1)
        return (images - mean) / std

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        value = self.normalize(images)
        value = F.relu(self.conv1(value))
        value = self.pool1(value)
        value = F.relu(self.conv2(value))
        value = self.pool2(value)
        value = F.relu(self.conv3(value))
        return self.conv4(value)


class EfficientADAutoencoder(nn.Module):
    """Autoencoder from EfficientAD Table 8 for 256 x 256 inputs."""

    def __init__(self, output_channels: int = 384) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 8),
        )
        self.decoder_convs = nn.ModuleList([
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
        ])
        self.final_conv = nn.Conv2d(64, 64, 3, padding=1)
        self.output = nn.Conv2d(64, output_channels, 3, padding=1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        value = self.encoder(PDNSmall.normalize(images))
        for size, convolution in zip((3, 8, 15, 32, 63, 127), self.decoder_convs):
            value = F.interpolate(value, size=(size, size), mode="bilinear", align_corners=False)
            value = self.dropout(F.relu(convolution(value)))
        value = F.interpolate(value, size=(64, 64), mode="bilinear", align_corners=False)
        return self.output(F.relu(self.final_conv(value)))


class EfficientAD(AnomalyDetector):
    """Paper-faithful EfficientAD-S training and inference."""

    _NELSON_TO_LOCAL = {
        "0.weight": "conv1.weight", "0.bias": "conv1.bias",
        "3.weight": "conv2.weight", "3.bias": "conv2.bias",
        "6.weight": "conv3.weight", "6.bias": "conv3.bias",
        "8.weight": "conv4.weight", "8.bias": "conv4.bias",
    }

    def __init__(
        self, *, teacher_weights_path: str | None = None,
        require_teacher_weights: bool = True, channels: int = 384,
        max_steps: int = 70_000, learning_rate: float = 1e-4,
        weight_decay: float = 1e-5, hard_quantile: float = 0.999,
        checkpoint_interval: int = 1_000,
    ) -> None:
        super().__init__()
        if channels < 1:
            raise ValueError("channels must be at least 1")
        if max_steps < 1:
            raise ValueError("max_steps must be at least 1")
        if checkpoint_interval < 1:
            raise ValueError("checkpoint_interval must be at least 1")
        if not 0 < hard_quantile <= 1:
            raise ValueError("hard_quantile must be in (0, 1]")
        self.teacher = PDNSmall(channels)
        self.student = PDNSmall(2 * channels)
        self.autoencoder = EfficientADAutoencoder(channels)
        self.teacher.requires_grad_(False)
        self.channels = channels
        self.max_steps = max_steps
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.hard_quantile = hard_quantile
        self.checkpoint_interval = checkpoint_interval
        self.teacher_source = None
        if teacher_weights_path:
            self._load_teacher(Path(teacher_weights_path))
        elif require_teacher_weights:
            raise FileNotFoundError(
                "EfficientAD-S requires the nelson1425 teacher_small.pth checkpoint"
            )
        self.register_buffer("teacher_mean", torch.zeros(1, channels, 1, 1))
        self.register_buffer("teacher_std", torch.ones(1, channels, 1, 1))
        self.register_buffer("map_st_q90", torch.tensor(0.0))
        self.register_buffer("map_st_q995", torch.tensor(1.0))
        self.register_buffer("map_ae_q90", torch.tensor(0.0))
        self.register_buffer("map_ae_q995", torch.tensor(1.0))
        self.register_buffer("fitted", torch.tensor(False))
        self.fit_summary: dict[str, Any] = {}

    def train(self, mode: bool = True):
        super().train(mode)
        self.teacher.eval()
        return self

    def _load_teacher(self, path: Path) -> None:
        if not path.is_file():
            raise FileNotFoundError(f"EfficientAD teacher weights are missing: {path}")
        try:
            checkpoint = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            checkpoint = torch.load(path, map_location="cpu")
        state = checkpoint.get("state_dict", checkpoint)
        if not isinstance(state, Mapping):
            raise TypeError("The nelson1425 teacher checkpoint is not a state dict")
        if set(state) == set(self._NELSON_TO_LOCAL):
            state = {self._NELSON_TO_LOCAL[key]: value for key, value in state.items()}
        expected = set(self.teacher.state_dict())
        if set(state) != expected:
            raise ValueError(
                "Teacher checkpoint is incompatible with nelson1425 PDN-S: "
                f"expected {sorted(expected)}, got {sorted(state)}"
            )
        self.teacher.load_state_dict(state, strict=True)
        self.teacher_source = str(path)

    @property
    def is_fitted(self) -> bool:
        return bool(self.fitted)

    def checkpoint_config(self) -> dict[str, Any]:
        return {
            "channels": self.channels,
            "max_steps": self.max_steps,
            "learning_rate": self.learning_rate,
            "weight_decay": self.weight_decay,
            "hard_quantile": self.hard_quantile,
            "checkpoint_interval": self.checkpoint_interval,
            "teacher_source": self.teacher_source,
        }

    def _teacher(self, images: torch.Tensor) -> torch.Tensor:
        value = self.teacher(images)
        return (value - self.teacher_mean) / self.teacher_std.clamp_min(1e-6)

    @staticmethod
    def _augment_autoencoder_input(images: torch.Tensor) -> torch.Tensor:
        factor = float(torch.empty(()).uniform_(0.8, 1.2))
        operation = int(torch.randint(3, ()).item())
        from torchvision.transforms import functional as transform_functional
        if operation == 0:
            return transform_functional.adjust_brightness(images, factor)
        if operation == 1:
            return transform_functional.adjust_contrast(images, factor)
        return transform_functional.adjust_saturation(images, factor)

    @torch.no_grad()
    def _raw_maps(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        teacher = self._teacher(images)
        student = self.student(images)
        autoencoder = self.autoencoder(images)
        map_st = (student[:, : self.channels] - teacher).square().mean(1, keepdim=True)
        map_ae = (
            student[:, self.channels :] - autoencoder
        ).square().mean(1, keepdim=True)
        return map_st, map_ae

    def _training_checkpoint(
        self, *, optimizer, scheduler, step: int, penalty_loader,
        train_epoch_generator_state=None, train_batches_into_epoch: int = 0,
    ) -> dict[str, Any]:
        penalty_state = None
        penalty_dataset = getattr(penalty_loader, "dataset", None)
        if hasattr(penalty_dataset, "state_dict"):
            penalty_state = penalty_dataset.state_dict()
        return {
            "model": self.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "step": step,
            "torch_rng_state": torch.get_rng_state(),
            "python_rng_state": random.getstate(),
            "train_epoch_generator_state": train_epoch_generator_state,
            "train_batches_into_epoch": train_batches_into_epoch,
            "cuda_rng_state": (
                torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
            ),
            "penalty_dataset_state": penalty_state,
        }

    @staticmethod
    def _save_training_checkpoint(checkpoint: dict[str, Any], path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + ".tmp")
        torch.save(checkpoint, temporary)
        temporary.replace(path)

    def fit(
        self, train_loader, *, device, validation_loader=None,
        penalty_loader=None, work_dir=None, resume=False,
    ) -> EfficientAD:
        if penalty_loader is None:
            raise ValueError("EfficientAD training requires an ImageNet penalty_loader")
        if validation_loader is None:
            raise ValueError("EfficientAD calibration requires a normal validation_loader")
        if self.max_steps < 1:
            raise ValueError("max_steps must be at least 1")
        device = torch.device(device)
        print(
            f"[EfficientAD] Starting fit: device={device}, max_steps={self.max_steps:,}, "
            f"resume={resume}, checkpoint_interval={self.checkpoint_interval:,}.",
            flush=True,
        )
        self.to(device)
        self.train(True)
        optimizer = torch.optim.Adam(
            [*self.student.parameters(), *self.autoencoder.parameters()],
            lr=self.learning_rate, weight_decay=self.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer, step_size=max(1, int(0.95 * self.max_steps)), gamma=0.1
        )
        checkpoint_path = (
            None if work_dir is None else Path(work_dir) / "efficientad_training.ckpt"
        )
        step = 0
        resume_train_epoch_state = None
        resume_train_batches = 0
        checkpoint_rng_state = None
        checkpoint_python_rng_state = None
        checkpoint_cuda_rng_state = None
        print(
            f"[EfficientAD] Training checkpoint: "
            f"{checkpoint_path if checkpoint_path is not None else 'disabled'}",
            flush=True,
        )
        if resume:
            if checkpoint_path is None or not checkpoint_path.is_file():
                raise FileNotFoundError(
                    "resume=true requires work_dir/efficientad_training.ckpt"
                )
            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
            self.load_state_dict(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            scheduler.load_state_dict(checkpoint["scheduler"])
            step = int(checkpoint["step"])
            resume_train_epoch_state = checkpoint.get("train_epoch_generator_state")
            resume_train_batches = int(checkpoint.get("train_batches_into_epoch", 0))
            if checkpoint.get("penalty_dataset_state") is not None:
                penalty_dataset = getattr(penalty_loader, "dataset", None)
                if not hasattr(penalty_dataset, "load_state_dict"):
                    raise TypeError("The penalty dataset cannot restore its stream state")
                penalty_dataset.load_state_dict(checkpoint["penalty_dataset_state"])
            checkpoint_rng_state = checkpoint["torch_rng_state"]
            checkpoint_python_rng_state = checkpoint.get("python_rng_state")
            checkpoint_cuda_rng_state = checkpoint.get("cuda_rng_state")
            print(
                f"[EfficientAD] Resume state restored at step {step:,}; "
                f"train batches into epoch={resume_train_batches:,}.",
                flush=True,
            )

        if step == 0:
            print(
                "[EfficientAD] Computing teacher feature mean/std on the clean "
                "training set...",
                flush=True,
            )
            sums = torch.zeros(self.channels, device=device)
            squares = torch.zeros_like(sums)
            count = 0
            with torch.no_grad():
                for batch in tqdm(
                    train_loader, desc="EfficientAD teacher statistics",
                    unit="batch", leave=False,
                ):
                    value = self.teacher(_clean_images(batch, device))
                    sums += value.sum((0, 2, 3))
                    squares += value.square().sum((0, 2, 3))
                    count += value.shape[0] * value.shape[2] * value.shape[3]
            if count == 0:
                raise ValueError("train_loader produced no images")
            mean = sums / count
            variance = (squares / count - mean.square()).clamp_min(1e-6)
            self.teacher_mean.copy_(mean.view(1, -1, 1, 1))
            self.teacher_std.copy_(variance.sqrt().view(1, -1, 1, 1))
            print(
                f"[EfficientAD] Teacher statistics ready from {count:,} feature "
                "vectors.",
                flush=True,
            )
        else:
            print(
                "[EfficientAD] Reusing teacher statistics from the resume checkpoint.",
                flush=True,
            )

        train_generator = getattr(train_loader, "generator", None)
        if resume_train_epoch_state is not None and train_generator is not None:
            train_generator.set_state(resume_train_epoch_state)
        train_epoch_generator_state = (
            train_generator.get_state() if train_generator is not None else None
        )
        train_iterator = iter(train_loader)
        train_batches_into_epoch = 0
        if resume_train_epoch_state is not None:
            for _ in range(resume_train_batches):
                try:
                    next(train_iterator)
                except StopIteration as error:
                    raise RuntimeError("Saved train-loader position is invalid") from error
                train_batches_into_epoch += 1
        penalty_iterator = iter(penalty_loader)
        penalty_batch_seen = False
        print(
            "[EfficientAD] ImageNet penalty iterator created. The first next() call "
            "will perform the initial remote read.",
            flush=True,
        )
        if checkpoint_rng_state is not None:
            torch.set_rng_state(checkpoint_rng_state)
            if checkpoint_python_rng_state is not None:
                random.setstate(checkpoint_python_rng_state)
            if torch.cuda.is_available() and checkpoint_cuda_rng_state is not None:
                torch.cuda.set_rng_state_all(checkpoint_cuda_rng_state)
        progress = tqdm(
            total=self.max_steps, initial=step, desc="EfficientAD-S training",
            unit="step", dynamic_ncols=True,
        )
        while step < self.max_steps:
            try:
                batch = next(train_iterator)
            except StopIteration:
                train_epoch_generator_state = (
                    train_generator.get_state() if train_generator is not None else None
                )
                train_batches_into_epoch = 0
                train_iterator = iter(train_loader)
                try:
                    batch = next(train_iterator)
                except StopIteration as error:
                    raise ValueError("train_loader produced no images") from error
            if not penalty_batch_seen:
                print(
                    "[EfficientAD] Waiting for the first ImageNet penalty batch...",
                    flush=True,
                )
            try:
                penalty_batch = next(penalty_iterator)
            except StopIteration:
                penalty_iterator = iter(penalty_loader)
                try:
                    penalty_batch = next(penalty_iterator)
                except StopIteration as error:
                    raise ValueError("penalty_loader produced no images") from error
            if not penalty_batch_seen:
                penalty_batch_seen = True
                print(
                    f"[EfficientAD] First ImageNet penalty batch ready with shape "
                    f"{tuple(penalty_batch['image'].shape)}. Training can proceed.",
                    flush=True,
                )

            train_batches_into_epoch += 1
            images = _clean_images(batch, device)
            penalty_images = penalty_batch["image"].to(device, non_blocking=True)
            with torch.no_grad():
                teacher = self._teacher(images)
            student = self.student(images)
            distance = (student[:, : self.channels] - teacher).square()
            threshold = torch.quantile(distance.detach(), self.hard_quantile)
            loss_hard = distance[distance >= threshold].mean()
            loss_penalty = self.student(penalty_images)[:, : self.channels].square().mean()

            augmented = self._augment_autoencoder_input(images)
            with torch.no_grad():
                teacher_augmented = self._teacher(augmented)
            autoencoder = self.autoencoder(augmented)
            student_autoencoder = self.student(augmented)[:, self.channels :]
            loss_autoencoder = F.mse_loss(autoencoder, teacher_augmented)
            loss_student_autoencoder = F.mse_loss(student_autoencoder, autoencoder)
            loss = (
                loss_hard + loss_penalty + loss_autoencoder
                + loss_student_autoencoder
            )
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            scheduler.step()
            step += 1
            progress.update(1)
            if step % 50 == 0 or step == self.max_steps:
                progress.set_postfix(
                    loss=f"{loss.detach().item():.4f}",
                    lr=f"{scheduler.get_last_lr()[0]:.2e}", refresh=False,
                )
            if (
                checkpoint_path is not None
                and (step % self.checkpoint_interval == 0 or step == self.max_steps)
            ):
                self._save_training_checkpoint(
                    self._training_checkpoint(
                        optimizer=optimizer, scheduler=scheduler, step=step,
                        penalty_loader=penalty_loader,
                        train_epoch_generator_state=train_epoch_generator_state,
                        train_batches_into_epoch=train_batches_into_epoch,
                    ),
                    checkpoint_path,
                )
                print(
                    f"[EfficientAD] Checkpoint saved at step {step:,}: "
                    f"{checkpoint_path}",
                    flush=True,
                )

        progress.close()
        print(
            "[EfficientAD] Optimization complete; starting clean-validation "
            "quantile calibration.",
            flush=True,
        )
        self.eval()
        st_values: list[torch.Tensor] = []
        ae_values: list[torch.Tensor] = []
        normal_images = 0
        with torch.no_grad():
            for batch in tqdm(
                validation_loader, desc="EfficientAD calibration",
                unit="batch", leave=False,
            ):
                labels = torch.as_tensor(batch.get("label", 0))
                clean = labels == 0
                if clean.any():
                    images = batch["image"][clean].to(device, non_blocking=True)
                    map_st, map_ae = self._raw_maps(images)
                    map_st = F.interpolate(
                        map_st, images.shape[-2:], mode="bilinear", align_corners=False
                    )
                    map_ae = F.interpolate(
                        map_ae, images.shape[-2:], mode="bilinear", align_corners=False
                    )
                    st_values.append(map_st.flatten().cpu())
                    ae_values.append(map_ae.flatten().cpu())
                    normal_images += images.shape[0]
        if not st_values:
            raise ValueError("calibration requires at least one normal validation image")
        st_tensor = torch.cat(st_values).float()
        ae_tensor = torch.cat(ae_values).float()
        st_q90 = torch.quantile(st_tensor, 0.90)
        st_q995 = torch.maximum(torch.quantile(st_tensor, 0.995), st_q90 + 1e-6)
        ae_q90 = torch.quantile(ae_tensor, 0.90)
        ae_q995 = torch.maximum(torch.quantile(ae_tensor, 0.995), ae_q90 + 1e-6)
        self.map_st_q90.copy_(st_q90.to(device))
        self.map_st_q995.copy_(st_q995.to(device))
        self.map_ae_q90.copy_(ae_q90.to(device))
        self.map_ae_q995.copy_(ae_q995.to(device))
        print(
            f"[EfficientAD] Calibration complete on {normal_images:,} normal images: "
            f"ST(q90={st_q90.item():.6g}, q99.5={st_q995.item():.6g}), "
            f"AE(q90={ae_q90.item():.6g}, q99.5={ae_q995.item():.6g}).",
            flush=True,
        )
        self.fitted.fill_(True)
        self.fit_summary = {
            "steps": step,
            "normal_calibration_images": normal_images,
            "calibration_pixels_per_map": st_tensor.numel(),
            "scheduler_step_size": max(1, int(0.95 * self.max_steps)),
        }
        if checkpoint_path is not None:
            self._save_training_checkpoint(
                self._training_checkpoint(
                    optimizer=optimizer, scheduler=scheduler, step=step,
                    penalty_loader=penalty_loader,
                    train_epoch_generator_state=train_epoch_generator_state,
                    train_batches_into_epoch=train_batches_into_epoch,
                ),
                checkpoint_path,
            )
        return self

    @staticmethod
    def _calibrate_map(
        value: torch.Tensor, q90: torch.Tensor, q995: torch.Tensor,
    ) -> torch.Tensor:
        return 0.1 * (value - q90) / (q995 - q90).clamp_min(1e-6)

    @torch.no_grad()
    def predict_with_raw(self, images: torch.Tensor):
        if not self.is_fitted:
            raise RuntimeError("EfficientAD must be fitted before predict")
        map_st, map_ae = self._raw_maps(images)
        map_st = F.interpolate(
            map_st, images.shape[-2:], mode="bilinear", align_corners=False
        )
        map_ae = F.interpolate(
            map_ae, images.shape[-2:], mode="bilinear", align_corners=False
        )
        anomaly_map = 0.5 * (
            self._calibrate_map(map_st, self.map_st_q90, self.map_st_q995)
            + self._calibrate_map(map_ae, self.map_ae_q90, self.map_ae_q995)
        )
        raw_map = anomaly_map
        bounded_map = anomaly_map.clamp(0, 1)
        score = bounded_map.flatten(1).amax(1)
        return AnomalyPrediction(score, bounded_map), raw_map.flatten(1).amax(1), raw_map

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        return self.predict_with_raw(images)[0]

def build_explicit_torchvision_backbone(
    backbone: str, *, pretrained: bool, weights_name: str,
) -> nn.Module:
    weights = None
    if pretrained:
        weights_enum = get_model_weights(backbone)
        try:
            weights = getattr(weights_enum, weights_name)
        except AttributeError as error:
            raise ValueError(
                f"Unknown torchvision weights {weights_name!r} for {backbone!r}"
            ) from error
    return get_model(backbone, weights=weights)


def _init_supersimplenet_weights(module: nn.Module) -> None:
    if isinstance(module, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_normal_(module.weight)
    elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.constant_(module.weight, 1)


def _rand_perlin_2d(
    shape: tuple[int, int], res: tuple[int, int], *, device: torch.device,
) -> torch.Tensor:
    """Torch port of the Perlin generator used by official SuperSimpleNet."""
    delta = (res[0] / shape[0], res[1] / shape[1])
    repeats = (shape[0] // res[0], shape[1] // res[1])
    y = torch.arange(0, res[0], delta[0], device=device)
    x = torch.arange(0, res[1], delta[1], device=device)
    grid = torch.stack(torch.meshgrid(y, x, indexing="ij"), dim=-1) % 1
    angles = 2 * math.pi * torch.rand(res[0] + 1, res[1] + 1, device=device)
    gradients = torch.stack((torch.cos(angles), torch.sin(angles)), dim=-1)

    def tiled(y_slice, x_slice):
        return (
            gradients[y_slice[0]:y_slice[1], x_slice[0]:x_slice[1]]
            .repeat_interleave(repeats[0], 0)
            .repeat_interleave(repeats[1], 1)
        )

    def dot(gradient, shift):
        offsets = torch.stack(
            (grid[:shape[0], :shape[1], 0] + shift[0],
             grid[:shape[0], :shape[1], 1] + shift[1]),
            dim=-1,
        )
        return (offsets * gradient[:shape[0], :shape[1]]).sum(dim=-1)

    n00 = dot(tiled((0, -1), (0, -1)), (0, 0))
    n10 = dot(tiled((1, None), (0, -1)), (-1, 0))
    n01 = dot(tiled((0, -1), (1, None)), (0, -1))
    n11 = dot(tiled((1, None), (1, None)), (-1, -1))
    fade = lambda value: 6 * value**5 - 15 * value**4 + 10 * value**3
    blend = fade(grid[:shape[0], :shape[1]])
    return math.sqrt(2) * torch.lerp(
        torch.lerp(n00, n10, blend[..., 0]),
        torch.lerp(n01, n11, blend[..., 0]),
        blend[..., 1],
    )


def _ssn_focal_loss(
    probabilities: torch.Tensor, targets: torch.Tensor, gamma: float = 4.0,
    reduction: str | None = "mean",
) -> torch.Tensor:
    probabilities = probabilities.float()
    targets = targets.float()
    ce = F.binary_cross_entropy(probabilities, targets, reduction="none")
    pt = probabilities * targets + (1 - probabilities) * (1 - targets)
    loss = ce * (1 - pt).pow(gamma)
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss


class SuperSimpleFeatureExtractor(nn.Module):
    def __init__(
        self, *, backbone: str, pretrained: bool, weights_name: str,
        layers: tuple[str, ...], patch_size: int, input_size: tuple[int, int],
    ) -> None:
        super().__init__()
        model = build_explicit_torchvision_backbone(
            backbone, pretrained=pretrained, weights_name=weights_name
        )
        self.extractor = create_feature_extractor(
            model, return_nodes={layer: layer for layer in layers}
        )
        self.extractor.requires_grad_(False)
        self.layers = layers
        self.pooler = nn.AvgPool2d(patch_size, stride=1, padding=patch_size // 2)
        with torch.no_grad():
            outputs = self.extractor(torch.zeros(1, 3, *input_size))
        self.feature_channels = sum(value.shape[1] for value in outputs.values())

    def train(self, mode: bool = True):
        super().train(False)
        return self

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        self.extractor.eval()
        with torch.no_grad():
            features = list(self.extractor(images).values())
        height, width = features[0].shape[-2:]
        features = [
            F.interpolate(
                feature, size=(height * 2, width * 2),
                mode="bilinear", align_corners=True,
            )
            for feature in features
        ]
        return self.pooler(torch.cat(features, dim=1))


class SuperSimpleFeatureAdaptor(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.projection = nn.Conv2d(channels, channels, 1)
        self.apply(_init_supersimplenet_weights)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.projection(features)


class SuperSimpleDiscriminator(nn.Module):
    def __init__(self, channels: int, *, hidden_channels: int, stop_grad: bool) -> None:
        super().__init__()
        self.stop_grad = stop_grad
        self.segmentor = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, 1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(hidden_channels, 1, 1, bias=False),
        )
        self.decision_features = nn.Sequential(
            nn.Conv2d(channels + 1, 128, 5, padding="same"),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.score = nn.Linear(128 * 2 + 2, 1)
        self.apply(_init_supersimplenet_weights)

    def parameter_groups(self):
        return (
            self.segmentor.parameters(),
            [*self.decision_features.parameters(), *self.score.parameters()],
        )

    def forward(
        self, segmentation_features: torch.Tensor,
        classification_features: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        anomaly_map = self.segmentor(segmentation_features)
        decision_map = anomaly_map.detach() if self.stop_grad else anomaly_map
        decision = self.decision_features(
            torch.cat((classification_features, decision_map), dim=1)
        )
        map_for_pooling = decision_map if self.stop_grad else anomaly_map
        pooled = torch.cat((
            F.adaptive_max_pool2d(decision, 1),
            F.adaptive_avg_pool2d(decision, 1),
            F.adaptive_max_pool2d(map_for_pooling, 1),
            F.adaptive_avg_pool2d(map_for_pooling, 1),
        ), dim=1).flatten(1)
        return anomaly_map, self.score(pooled).flatten()


class SuperSimpleAnomalyGenerator(nn.Module):
    def __init__(
        self, *, noise_std: float, threshold: float,
        perlin_range: tuple[int, int] = (0, 6),
    ) -> None:
        super().__init__()
        self.noise_std = noise_std
        self.threshold = threshold
        self.perlin_range = perlin_range

    def _masks(
        self, batch_size: int, height: int, width: int, device: torch.device,
    ) -> torch.Tensor:
        power_height = 1 << (height - 1).bit_length()
        power_width = 1 << (width - 1).bit_length()
        masks = []
        for _ in range(batch_size):
            scale_y = 2 ** int(torch.randint(*self.perlin_range, (1,)).item())
            scale_x = 2 ** int(torch.randint(*self.perlin_range, (1,)).item())
            noise = _rand_perlin_2d(
                (power_height, power_width), (scale_y, scale_x), device=device
            )
            noise = F.interpolate(
                noise[None, None], size=(height, width),
                mode="bilinear", align_corners=False,
            )[0]
            mask = (noise > self.threshold).float()
            if torch.rand(()) > 0.5:
                mask.zero_()  # official no_anomaly="empty"
            masks.append(mask)
        return torch.stack(masks)

    def forward(
        self, features: torch.Tensor, adapted: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        batch, _, height, width = adapted.shape
        features = torch.cat((features, features), dim=0)
        adapted = torch.cat((adapted, adapted), dim=0)
        mask = self._masks(batch * 2, height, width, adapted.device)
        noise = torch.normal(0, self.noise_std, size=adapted.shape, device=adapted.device)
        return features + noise * mask, adapted + noise * mask, mask


class SuperSimpleNet(AnomalyDetector):
    """JIMS SuperSimpleNet in the official unsupervised MVTec configuration."""

    def __init__(
        self, *, backbone: str = "wide_resnet50_2",
        pretrained: bool = True, weights_name: str = "IMAGENET1K_V1",
        layers: tuple[str, ...] = ("layer2", "layer3"),
        input_size: tuple[int, int] = (256, 256), patch_size: int = 3,
        epochs: int = 300, noise_std: float = 0.015,
        perlin_threshold: float = 0.2, adaptor_learning_rate: float = 1e-4,
        segmentation_learning_rate: float = 2e-4,
        decision_learning_rate: float = 2e-4, scheduler_gamma: float = 0.4,
        stop_grad: bool = True, adapt_classification_features: bool = False,
        gradient_clip: bool = False, margin: float = 0.5,
        gaussian_sigma: float = 4.0, fixed_training_duration: bool = True,
        validation_interval: int = 4, validation_batches: int = 64,
        max_samples_per_epoch: int | None = None,
    ) -> None:
        super().__init__()
        self.backbone_name = backbone
        self.pretrained = pretrained
        self.weights_name = weights_name
        self.layers = tuple(layers)
        self.input_size = tuple(input_size)
        self.patch_size = patch_size
        self.epochs = epochs
        self.noise_std = noise_std
        self.perlin_threshold = perlin_threshold
        self.adaptor_learning_rate = adaptor_learning_rate
        self.segmentation_learning_rate = segmentation_learning_rate
        self.decision_learning_rate = decision_learning_rate
        self.scheduler_gamma = scheduler_gamma
        self.stop_grad = stop_grad
        self.adapt_classification_features = adapt_classification_features
        self.gradient_clip = gradient_clip
        self.margin = margin
        self.gaussian_sigma = gaussian_sigma
        self.fixed_training_duration = fixed_training_duration
        self.validation_interval = validation_interval
        self.validation_batches = validation_batches
        self.max_samples_per_epoch = max_samples_per_epoch

        self.features = SuperSimpleFeatureExtractor(
            backbone=backbone, pretrained=pretrained, weights_name=weights_name,
            layers=self.layers, patch_size=patch_size, input_size=self.input_size,
        )
        channels = self.features.feature_channels
        self.adaptor = SuperSimpleFeatureAdaptor(channels)
        self.discriminator = SuperSimpleDiscriminator(
            channels, hidden_channels=1024, stop_grad=stop_grad
        )
        self.anomaly_generator = SuperSimpleAnomalyGenerator(
            noise_std=noise_std, threshold=perlin_threshold
        )
        self.register_buffer("fitted", torch.tensor(False))
        self.fit_summary: dict[str, Any] = {}

    @property
    def is_fitted(self) -> bool:
        return bool(self.fitted)

    def train(self, mode: bool = True):
        super().train(mode)
        self.features.eval()
        return self

    def checkpoint_config(self) -> dict[str, Any]:
        return {
            "backbone": self.backbone_name, "pretrained": self.pretrained,
            "weights_name": self.weights_name, "layers": list(self.layers),
            "input_size": list(self.input_size), "patch_size": self.patch_size,
            "epochs": self.epochs, "noise_std": self.noise_std,
            "perlin_threshold": self.perlin_threshold,
            "adaptor_learning_rate": self.adaptor_learning_rate,
            "segmentation_learning_rate": self.segmentation_learning_rate,
            "decision_learning_rate": self.decision_learning_rate,
            "scheduler_gamma": self.scheduler_gamma, "stop_grad": self.stop_grad,
            "adapt_classification_features": self.adapt_classification_features,
            "gradient_clip": self.gradient_clip, "margin": self.margin,
            "gaussian_sigma": self.gaussian_sigma,
            "fixed_training_duration": self.fixed_training_duration,
            "validation_interval": self.validation_interval,
            "validation_batches": self.validation_batches,
            "max_samples_per_epoch": self.max_samples_per_epoch,
        }

    def _logits(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        features = self.features(images)
        adapted = self.adaptor(features)
        classification = adapted if self.adapt_classification_features else features
        return self.discriminator(adapted, classification)

    def fit(
        self, train_loader, *, device, validation_loader=None,
        work_dir=None, resume=False,
    ) -> SuperSimpleNet:
        if resume:
            raise NotImplementedError("SuperSimpleNet resume is not implemented")
        device = torch.device(device)
        self.to(device)
        segmentation_parameters, decision_parameters = self.discriminator.parameter_groups()
        optimizer = torch.optim.AdamW([
            {"params": self.adaptor.parameters(), "lr": self.adaptor_learning_rate},
            {"params": segmentation_parameters, "lr": self.segmentation_learning_rate,
             "weight_decay": 1e-5},
            {"params": decision_parameters, "lr": self.decision_learning_rate,
             "weight_decay": 1e-5},
        ])
        scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer,
            milestones=[int(self.epochs * 0.8), int(self.epochs * 0.9)],
            gamma=self.scheduler_gamma,
        )
        best_auc = -math.inf
        best_state = None
        steps = 0
        epoch_progress = tqdm(
            range(1, self.epochs + 1), desc="SuperSimpleNet training",
            unit="epoch", dynamic_ncols=True,
        )
        for epoch in epoch_progress:
            self.train(True)
            samples = 0
            produced = False
            for batch in train_loader:
                produced = True
                images = _clean_images(batch, device)
                with torch.no_grad():
                    features = self.features(images)
                adapted = self.adaptor(features)
                noisy_features, noisy_adapted, target_mask = self.anomaly_generator(
                    features, adapted
                )
                classification = (
                    noisy_adapted if self.adapt_classification_features
                    else noisy_features
                )
                anomaly_map, score = self.discriminator(noisy_adapted, classification)
                target_label = target_mask.flatten(1).amax(1)

                focal = _ssn_focal_loss(
                    torch.sigmoid(anomaly_map), target_mask, reduction=None
                )
                truncated = torch.zeros_like(anomaly_map)
                normal = target_mask == 0
                anomalous = target_mask > 0
                truncated[normal] = torch.clamp(anomaly_map[normal] + self.margin, min=0)
                truncated[anomalous] = torch.clamp(-anomaly_map[anomalous] + self.margin, min=0)
                good_loss = truncated[normal].mean() if normal.any() else 0.0
                bad_loss = truncated[anomalous].mean() if anomalous.any() else 0.0
                loss = good_loss + bad_loss + focal.mean()
                loss = loss + _ssn_focal_loss(torch.sigmoid(score), target_label)

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if self.gradient_clip:
                    torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
                optimizer.step()
                epoch_progress.set_postfix(
                    loss=f"{loss.detach().item():.4f}", refresh=False
                )
                steps += 1
                samples += images.shape[0]
                if self.max_samples_per_epoch and samples >= self.max_samples_per_epoch:
                    break
            if not produced:
                raise ValueError("train_loader produced no images")
            scheduler.step()
            if (
                not self.fixed_training_duration
                and (epoch % self.validation_interval == 0 or epoch == self.epochs)
            ):
                self.fitted.fill_(True)
                auc = _validation_auc(
                    self, validation_loader, device, self.validation_batches
                )
                if not math.isnan(auc) and auc > best_auc:
                    best_auc = auc
                    best_state = copy.deepcopy(self.state_dict())
        if best_state is not None:
            self.load_state_dict(best_state)
        self.fitted.fill_(True)
        self.fit_summary = {
            "epochs": self.epochs, "steps": steps,
            "fixed_training_duration": self.fixed_training_duration,
            "selected_validation_image_auroc": (
                None if self.fixed_training_duration or best_state is None else best_auc
            ),
        }
        return self

    @torch.no_grad()
    def predict_with_raw(self, images: torch.Tensor):
        if not self.is_fitted:
            raise RuntimeError("SuperSimpleNet must be fitted before predict")
        self.eval()
        raw_map, raw_score = self._logits(images)
        raw_map = F.interpolate(
            raw_map, images.shape[-2:], mode="bilinear", align_corners=False
        )
        if self.gaussian_sigma > 0:
            kernel = 2 * math.ceil(3 * self.gaussian_sigma) + 1
            raw_map = gaussian_blur(
                raw_map, [kernel, kernel], [self.gaussian_sigma, self.gaussian_sigma]
            )
        anomaly_map = torch.sigmoid(raw_map)
        anomaly_score = torch.sigmoid(raw_score)
        prediction = AnomalyPrediction(anomaly_score, anomaly_map)
        return prediction, raw_score, raw_map

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        return self.predict_with_raw(images)[0]

class TinyGLASSFeatureExtractor(nn.Module):
    """TinyGLASS ResNet-18 layer2/layer3 patch-grid embedding."""

    def __init__(
        self, *, pretrained: bool, weights_name: str,
        patch_size: int = 3, output_channels_per_layer: int = 64,
    ) -> None:
        super().__init__()
        backbone = build_explicit_torchvision_backbone(
            "resnet18", pretrained=pretrained, weights_name=weights_name
        )
        self.extractor = create_feature_extractor(
            backbone, return_nodes={"layer2": "layer2", "layer3": "layer3"}
        )
        self.extractor.requires_grad_(False)
        self.patch_size = patch_size
        self.output_channels_per_layer = output_channels_per_layer

    def train(self, mode: bool = True):
        super().train(False)
        return self

    def _patch_grid(self, feature: torch.Tensor) -> torch.Tensor:
        patches = F.unfold(
            feature, kernel_size=self.patch_size, stride=1,
            padding=self.patch_size // 2,
        )
        return patches.reshape(
            feature.shape[0], -1, feature.shape[-2], feature.shape[-1]
        )

    def _reduce(self, feature: torch.Tensor) -> torch.Tensor:
        channels = feature.shape[1]
        if channels % self.output_channels_per_layer:
            raise ValueError(
                f"Patch channels {channels} are not divisible by "
                f"{self.output_channels_per_layer}"
            )
        return feature.reshape(
            feature.shape[0], self.output_channels_per_layer,
            channels // self.output_channels_per_layer,
            *feature.shape[-2:],
        ).mean(2)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        self.extractor.eval()
        with torch.no_grad():
            features = self.extractor(images)
        layer2 = self._patch_grid(features["layer2"])
        layer3 = self._patch_grid(features["layer3"])
        layer2 = F.adaptive_avg_pool2d(layer2, layer3.shape[-2:])
        return torch.cat((self._reduce(layer2), self._reduce(layer3)), dim=1)


class TinyGLASSDiscriminator(nn.Module):
    def __init__(self, input_channels: int = 128, hidden_channels: int = 512) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(input_channels, hidden_channels, 1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(0.2),
            nn.Conv2d(hidden_channels, 1, 1, bias=False),
            nn.Sigmoid(),
        )
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, nn.Conv2d):
            nn.init.normal_(module.weight, 0.0, 0.02)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.normal_(module.weight, 1.0, 0.02)
            nn.init.zeros_(module.bias)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features).squeeze(1)


class DTDTextureSampler:
    """Texture loader and random augmentation used by TinyGLASS LAS."""

    _EXTENSIONS = {".jpg", ".jpeg", ".png"}

    def __init__(self, root: str | None, input_size: tuple[int, int]) -> None:
        self.root = None if root is None else Path(root)
        self.input_size = input_size
        self.paths = [] if self.root is None else sorted(
            path for path in self.root.rglob("*")
            if path.suffix.lower() in self._EXTENSIONS
        )

    @staticmethod
    def _random_operations(image: Image.Image) -> Image.Image:
        operations = [
            lambda value: TF.adjust_contrast(value, random.uniform(0.8, 1.2)),
            lambda value: TF.adjust_brightness(value, random.uniform(0.8, 1.2)),
            lambda value: TF.adjust_hue(
                TF.adjust_saturation(value, random.uniform(0.8, 1.2)),
                random.uniform(-0.2, 0.2),
            ),
            TF.hflip,
            TF.vflip,
            lambda value: TF.rgb_to_grayscale(value, num_output_channels=3),
            TF.autocontrast,
            TF.equalize,
            lambda value: TF.rotate(
                value, random.uniform(-45, 45),
                interpolation=InterpolationMode.BILINEAR,
            ),
        ]
        for index in random.sample(range(len(operations)), 3):
            image = operations[index](image)
        return image

    def sample(self, batch_size: int, *, device: torch.device) -> torch.Tensor:
        if not self.paths:
            raise RuntimeError("TinyGLASS requires DTD textures for LAS")
        values = []
        for index in torch.randint(len(self.paths), (batch_size,)).tolist():
            with Image.open(self.paths[index]) as image:
                image = TF.resize(
                    image.convert("RGB"), list(self.input_size),
                    interpolation=InterpolationMode.BILINEAR, antialias=True,
                )
                image = self._random_operations(image)
                tensor = pil_to_tensor(image).float().div_(255.0)
            mean = tensor.new_tensor((0.485, 0.456, 0.406)).view(3, 1, 1)
            std = tensor.new_tensor((0.229, 0.224, 0.225)).view(3, 1, 1)
            values.append((tensor - mean) / std)
        return torch.stack(values).to(device, non_blocking=True)


class TinyGLASSLAS(nn.Module):
    def __init__(
        self, *, input_size: tuple[int, int], blend_mean: float = 0.5,
        blend_std: float = 0.1,
    ) -> None:
        super().__init__()
        self.input_size = input_size
        self.blend_mean = blend_mean
        self.blend_std = blend_std

    def _one_mask(self, device: torch.device) -> torch.Tensor:
        height, width = self.input_size
        for _ in range(32):
            masks = []
            for _ in range(2):
                scale_y = 2 ** int(torch.randint(0, 6, (1,)).item())
                scale_x = 2 ** int(torch.randint(0, 6, (1,)).item())
                noise = _rand_perlin_2d(
                    (height, width), (scale_y, scale_x), device=device
                )
                angle = float(torch.empty(()).uniform_(-90, 90))
                noise = TF.rotate(
                    noise[None], angle,
                    interpolation=InterpolationMode.BILINEAR,
                )[0]
                masks.append((noise > 0.5).float())
            choice = float(torch.rand(()))
            if choice > 2 / 3:
                mask = torch.clamp(masks[0] + masks[1], 0, 1)
            elif choice > 1 / 3:
                mask = masks[0] * masks[1]
            else:
                mask = masks[0]
            if mask.any():
                return mask.unsqueeze(0)
        raise RuntimeError("Could not generate a non-empty TinyGLASS Perlin mask")

    def forward(
        self, images: torch.Tensor, textures: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        masks = torch.stack([
            self._one_mask(images.device) for _ in range(images.shape[0])
        ])
        beta = torch.normal(
            self.blend_mean, self.blend_std, size=(images.shape[0], 1, 1, 1),
            device=images.device,
        ).clamp(0.2, 0.8)
        augmented = (
            images * (1 - masks)
            + ((1 - beta) * textures + beta * images) * masks
        )
        return augmented, masks


def _tinyglass_focal_loss(
    probabilities: torch.Tensor, targets: torch.Tensor,
    gamma: float = 2.0,
) -> torch.Tensor:
    probabilities = probabilities.clamp(1e-5, 1 - 1e-5)
    targets = targets.float()
    pt = probabilities * targets + (1 - probabilities) * (1 - targets)
    return (-(1 - pt).pow(gamma) * torch.log(pt)).mean()


class TinyGLASSDeployment(nn.Module):
    def __init__(
        self, features: TinyGLASSFeatureExtractor,
        discriminator: TinyGLASSDiscriminator,
    ) -> None:
        super().__init__()
        self.features = copy.deepcopy(features)
        self.discriminator = copy.deepcopy(discriminator)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.discriminator(self.features(images)).unsqueeze(1)


class TinyGLASS(AnomalyDetector):
    """TinyGLASS with dedicated LAS, GAS, patch embedding and discriminator."""

    def __init__(
        self, *, pretrained: bool = True,
        weights_name: str = "IMAGENET1K_V1",
        input_size: tuple[int, int] = (256, 256), patch_size: int = 3,
        epochs: int = 640, learning_rate: float = 1e-4,
        weight_decay: float = 1e-2, noise_std: float = 0.015,
        radius_quantile: float = 0.75, hard_mining_quantile: float = 0.5,
        gas_steps: int = 20, gas_step_size: float = 0.001,
        hypersphere_projection: bool = True,
        max_samples_per_epoch: int | None = 392,
        texture_root: str | None = None,
        require_texture_dataset: bool = True,
        blend_mean: float = 0.5, blend_std: float = 0.1,
        gaussian_sigma: float = 4.0,
        fixed_training_duration: bool = True,
        validation_interval: int = 1, validation_batches: int = 64,
    ) -> None:
        super().__init__()
        if require_texture_dataset and not texture_root:
            raise FileNotFoundError("TinyGLASS requires texture_root=dtd/images")
        if not 0 < radius_quantile < 1:
            raise ValueError("radius_quantile must be in (0, 1)")
        if not 0 <= hard_mining_quantile < 1:
            raise ValueError("hard_mining_quantile must be in [0, 1)")
        self.pretrained = pretrained
        self.weights_name = weights_name
        self.input_size = tuple(input_size)
        self.patch_size = patch_size
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.noise_std = noise_std
        self.radius_quantile = radius_quantile
        self.hard_mining_quantile = hard_mining_quantile
        self.gas_steps = gas_steps
        self.gas_step_size = gas_step_size
        self.hypersphere_projection = hypersphere_projection
        self.max_samples_per_epoch = max_samples_per_epoch
        self.require_texture_dataset = require_texture_dataset
        self.blend_mean = blend_mean
        self.blend_std = blend_std
        self.gaussian_sigma = gaussian_sigma
        self.fixed_training_duration = fixed_training_duration
        self.validation_interval = validation_interval
        self.validation_batches = validation_batches

        self.features = TinyGLASSFeatureExtractor(
            pretrained=pretrained, weights_name=weights_name,
            patch_size=patch_size,
        )
        self.discriminator = TinyGLASSDiscriminator()
        self.textures = DTDTextureSampler(texture_root, self.input_size)
        if require_texture_dataset and not self.textures.paths:
            raise FileNotFoundError(f"No DTD images found under {texture_root}")
        self.las = TinyGLASSLAS(
            input_size=self.input_size,
            blend_mean=blend_mean, blend_std=blend_std,
        )
        self.register_buffer("center", torch.zeros(128))
        self.register_buffer("fitted", torch.tensor(False))
        self.fit_summary: dict[str, Any] = {}

    @property
    def is_fitted(self) -> bool:
        return bool(self.fitted)

    def train(self, mode: bool = True):
        super().train(mode)
        self.features.eval()
        return self

    def checkpoint_config(self) -> dict[str, Any]:
        return {
            "backbone": "resnet18", "pretrained": self.pretrained,
            "weights_name": self.weights_name, "input_size": list(self.input_size),
            "patch_size": self.patch_size, "target_embedding_dimension": 128,
            "epochs": self.epochs, "learning_rate": self.learning_rate,
            "weight_decay": self.weight_decay, "noise_std": self.noise_std,
            "radius_quantile": self.radius_quantile,
            "hard_mining_quantile": self.hard_mining_quantile,
            "gas_steps": self.gas_steps, "gas_step_size": self.gas_step_size,
            "hypersphere_projection": self.hypersphere_projection,
            "max_samples_per_epoch": self.max_samples_per_epoch,
            "require_texture_dataset": self.require_texture_dataset,
            "blend_mean": self.blend_mean, "blend_std": self.blend_std,
            "gaussian_sigma": self.gaussian_sigma,
            "fixed_training_duration": self.fixed_training_duration,
            "validation_interval": self.validation_interval,
            "validation_batches": self.validation_batches,
        }

    @torch.no_grad()
    def _compute_center(self, train_loader, device: torch.device) -> int:
        total = torch.zeros_like(self.center, device=device)
        patches = 0
        for batch in tqdm(
            train_loader, desc="TinyGLASS feature center",
            unit="batch", leave=False,
        ):
            features = self.features(_clean_images(batch, device))
            flattened = features.permute(0, 2, 3, 1).reshape(-1, 128)
            total += flattened.sum(0)
            patches += flattened.shape[0]
        if patches == 0:
            raise ValueError("train_loader produced no images")
        self.center.copy_(total / patches)
        return patches

    def _project_gas(
        self, gas: torch.Tensor, true: torch.Tensor,
        center: torch.Tensor, radius: torch.Tensor,
    ) -> torch.Tensor:
        gas_flat = gas.permute(0, 2, 3, 1).reshape(-1, 128)
        true_flat = true.permute(0, 2, 3, 1).reshape(-1, 128)
        base = center if self.hypersphere_projection else true_flat
        lower = radius if self.hypersphere_projection else gas_flat.new_tensor(0.5)
        vector = gas_flat - base
        norm = torch.norm(vector, dim=1).clamp_min(1e-10)
        alpha = torch.clamp(norm, lower, 2 * lower)
        projected = base + vector * (alpha / norm).unsqueeze(1)
        return projected.reshape_as(gas.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

    def _training_loss(
        self, true_features: torch.Tensor, fake_features: torch.Tensor,
        feature_mask: torch.Tensor,
    ) -> torch.Tensor | None:
        batch, channels, height, width = true_features.shape
        true_flat = true_features.permute(0, 2, 3, 1).reshape(-1, channels)
        fake_flat = fake_features.permute(0, 2, 3, 1).reshape(-1, channels)
        mask_flat = feature_mask.reshape(-1).bool()
        center = self.center.reshape(1, -1).expand_as(true_flat)
        true_points = torch.cat((fake_flat[~mask_flat], true_flat), dim=0)
        true_centers = torch.cat((center[~mask_flat], center), dim=0)
        radius = torch.quantile(
            torch.norm(true_points - true_centers, dim=1), self.radius_quantile
        ).detach().clamp_min(1e-6)

        gas = (
            true_features + torch.normal(
                0, self.noise_std, size=true_features.shape,
                device=true_features.device,
            )
        ).detach().requires_grad_(True)
        bce_loss = None
        for gas_step in range(self.gas_steps + 1):
            true_scores = self.discriminator(true_features)
            gas_scores = self.discriminator(gas)
            bce_loss = (
                F.binary_cross_entropy(true_scores, torch.zeros_like(true_scores))
                + F.binary_cross_entropy(gas_scores, torch.ones_like(gas_scores))
            )
            if gas_step == self.gas_steps:
                break
            gradient = torch.autograd.grad(
                F.binary_cross_entropy(gas_scores, torch.ones_like(gas_scores)),
                gas,
            )[0]
            gradient = gradient / torch.norm(
                gradient, dim=1, keepdim=True
            ).clamp_min(1e-10)
            gas = (gas + self.gas_step_size * gradient).detach()
            if (gas_step + 1) % 5 == 0:
                gas = self._project_gas(gas, true_features, center, radius)
            gas.requires_grad_(True)

        if not mask_flat.any():
            return None
        if self.hypersphere_projection:
            anomalous = fake_flat[mask_flat]
            anomaly_center = center[mask_flat]
            vector = anomalous - anomaly_center
            norm = torch.norm(vector, dim=1).clamp_min(1e-10)
            alpha = torch.clamp(norm, 2 * radius, 4 * radius)
            fake_flat = fake_flat.clone()
            fake_flat[mask_flat] = anomaly_center + vector * (alpha / norm).unsqueeze(1)
            fake_features = fake_flat.reshape(
                batch, height, width, channels
            ).permute(0, 3, 1, 2)

        fake_scores = self.discriminator(fake_features)
        distance = (fake_scores - feature_mask.squeeze(1)).square()
        if self.hard_mining_quantile > 0:
            threshold = torch.quantile(distance.detach(), self.hard_mining_quantile)
            selected = distance >= threshold
            fake_scores = fake_scores[selected]
            target = feature_mask.squeeze(1)[selected]
        else:
            target = feature_mask.squeeze(1)
        return bce_loss + _tinyglass_focal_loss(fake_scores, target)

    def fit(
        self, train_loader, *, device, validation_loader=None,
        work_dir=None, resume=False,
    ) -> TinyGLASS:
        if resume:
            raise NotImplementedError("TinyGLASS resume is not implemented")
        device = torch.device(device)
        self.to(device)
        center_patches = self._compute_center(train_loader, device)
        optimizer = torch.optim.AdamW(
            self.discriminator.parameters(), lr=self.learning_rate * 2,
            weight_decay=self.weight_decay,
        )
        best_auc = -math.inf
        best_state = None
        steps = 0
        skipped_empty_masks = 0
        epoch_progress = tqdm(
            range(1, self.epochs + 1), desc="TinyGLASS training",
            unit="epoch", dynamic_ncols=True,
        )
        for epoch in epoch_progress:
            self.train(True)
            samples = 0
            produced = False
            for batch in train_loader:
                produced = True
                images = _clean_images(batch, device)
                textures = self.textures.sample(images.shape[0], device=device)
                augmented, image_mask = self.las(images, textures)
                with torch.no_grad():
                    true_features = self.features(images)
                    fake_features = self.features(augmented)
                feature_mask = F.interpolate(
                    image_mask, size=true_features.shape[-2:], mode="nearest"
                )
                loss = self._training_loss(
                    true_features.detach(), fake_features.detach(), feature_mask
                )
                if loss is None:
                    skipped_empty_masks += 1
                    continue
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                epoch_progress.set_postfix(
                    loss=f"{loss.detach().item():.4f}", refresh=False
                )
                steps += 1
                samples += images.shape[0]
                if self.max_samples_per_epoch and samples >= self.max_samples_per_epoch:
                    break
            if not produced:
                raise ValueError("train_loader produced no images")
            if (
                not self.fixed_training_duration
                and (epoch % self.validation_interval == 0 or epoch == self.epochs)
            ):
                self.fitted.fill_(True)
                auc = _validation_auc(
                    self, validation_loader, device, self.validation_batches
                )
                if not math.isnan(auc) and auc > best_auc:
                    best_auc = auc
                    best_state = copy.deepcopy(self.state_dict())
        if best_state is not None:
            self.load_state_dict(best_state)
        self.fitted.fill_(True)
        self.fit_summary = {
            "epochs": self.epochs, "steps": steps,
            "center_patches": center_patches,
            "skipped_empty_masks": skipped_empty_masks,
            "fixed_training_duration": self.fixed_training_duration,
            "selected_validation_image_auroc": (
                None if self.fixed_training_duration or best_state is None else best_auc
            ),
        }
        return self

    @torch.no_grad()
    def _patch_scores(self, images: torch.Tensor) -> torch.Tensor:
        return self.discriminator(self.features(images)).unsqueeze(1)

    @torch.no_grad()
    def predict_with_raw(self, images: torch.Tensor):
        if not self.is_fitted:
            raise RuntimeError("TinyGLASS must be fitted before predict")
        self.eval()
        patch_scores = self._patch_scores(images)
        anomaly_map = F.interpolate(
            patch_scores, images.shape[-2:], mode="bilinear", align_corners=False
        )
        if self.gaussian_sigma > 0:
            kernel = 2 * math.ceil(3 * self.gaussian_sigma) + 1
            anomaly_map = gaussian_blur(
                anomaly_map, [kernel, kernel],
                [self.gaussian_sigma, self.gaussian_sigma],
            )
        anomaly_score = patch_scores.flatten(1).amax(1)
        prediction = AnomalyPrediction(
            anomaly_score.clamp(0, 1), anomaly_map.clamp(0, 1)
        )
        return prediction, anomaly_score, patch_scores

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        return self.predict_with_raw(images)[0]

    def to_deployment_module(self) -> nn.Module:
        if not self.is_fitted:
            raise RuntimeError("TinyGLASS must be fitted before export")
        return TinyGLASSDeployment(self.features, self.discriminator).eval()

def build_model():
    if MODEL_NAME == "patchcore":
        return PatchCore(
            backbone=PATCHCORE_BACKBONE, pretrained=PATCHCORE_PRETRAINED,
            layers=PATCHCORE_LAYERS,
            coreset_sampling_ratio=PATCHCORE_CORESET_SAMPLING_RATIO,
            num_neighbors=PATCHCORE_NUM_NEIGHBORS,
            patch_size=PATCHCORE_PATCH_SIZE, patch_stride=PATCHCORE_PATCH_STRIDE,
            pretrain_embed_dimension=PATCHCORE_PRETRAIN_EMBED_DIMENSION,
            target_embed_dimension=PATCHCORE_TARGET_EMBED_DIMENSION,
            max_patches_per_image=PATCHCORE_MAX_PATCHES_PER_IMAGE,
            max_training_embeddings=PATCHCORE_MAX_TRAINING_EMBEDDINGS,
            max_memory_bank_size=PATCHCORE_MAX_MEMORY_BANK_SIZE,
            projection_dim=PATCHCORE_PROJECTION_DIM,
            sampling_seed=PATCHCORE_SAMPLING_SEED,
            calibration_quantile=PATCHCORE_CALIBRATION_QUANTILE,
            calibration_batches=PATCHCORE_CALIBRATION_BATCHES,
            distance_query_chunk_size=PATCHCORE_DISTANCE_QUERY_CHUNK_SIZE,
            distance_bank_chunk_size=PATCHCORE_DISTANCE_BANK_CHUNK_SIZE,
            gaussian_sigma=PATCHCORE_GAUSSIAN_SIGMA,
        )
    if MODEL_NAME == "efficientad_s":
        return EfficientAD(**{
            key: value for key, value in SELECTED_MODEL_CONFIG.items()
            if key != "batch_size"
        })
    if MODEL_NAME == "supersimplenet":
        return SuperSimpleNet(**{
            key: value for key, value in SELECTED_MODEL_CONFIG.items()
            if key != "batch_size"
        })
    if MODEL_NAME == "tinyglass":
        return TinyGLASS(**{
            key: value for key, value in SELECTED_MODEL_CONFIG.items()
            if key != "batch_size"
        })
    raise ValueError(f"Unknown model name: {MODEL_NAME}")

def build_optimizer(model):
    # I modelli possiedono il proprio fit; PatchCore resta non parametrico.
    return None

def build_scheduler(optimizer):
    return None


In [ ]:
class ExactBinaryMetrics:
    # Image-level evaluation retains only one score and label per image.
    def __init__(self):
        self._scores = []
        self._targets = []

    @torch.no_grad()
    def update(self, scores, targets):
        scores = scores.detach().reshape(-1).cpu()
        targets = targets.detach().reshape(-1).cpu().bool()
        if scores.numel() != targets.numel():
            raise ValueError("scores and targets must contain the same number of values")
        if not scores.numel():
            return
        if not scores.is_floating_point() or not torch.isfinite(scores).all():
            raise ValueError("scores must contain finite floating-point values")
        if scores.min() < 0 or scores.max() > 1:
            raise ValueError("scores must be normalized to [0, 1]")
        self._scores.append(scores.clone())
        self._targets.append(targets.clone())

    def compute(self):
        if not self._scores:
            raise ValueError("AUROC and average precision require both target classes")
        scores = torch.cat(self._scores)
        targets = torch.cat(self._targets)
        positives = int(targets.sum())
        negatives = targets.numel() - positives
        if positives == 0 or negatives == 0:
            raise ValueError("AUROC and average precision require both target classes")
        order = torch.argsort(scores, descending=True)
        sorted_scores = scores[order]
        sorted_targets = targets[order]
        _, group_counts = torch.unique_consecutive(sorted_scores, return_counts=True)
        group_ends = group_counts.cumsum(0) - 1
        true_positives = sorted_targets.cumsum(0)[group_ends].double()
        false_positives = (~sorted_targets).cumsum(0)[group_ends].double()
        recall = true_positives / positives
        false_positive_rate = false_positives / negatives
        precision = true_positives / (true_positives + false_positives)
        zero = torch.zeros(1, dtype=torch.float64)
        auroc = torch.trapezoid(
            torch.cat((zero, recall)), torch.cat((zero, false_positive_rate))
        )
        recall_increment = recall - torch.cat((zero, recall[:-1]))
        average_precision = (precision * recall_increment).sum()
        return {"auroc": float(auroc), "average_precision": float(average_precision)}


class BinaryHistogramMetrics:
    # Fixed-size histograms avoid retaining every pixel score in memory.
    def __init__(self, num_bins=2048):
        if num_bins < 2:
            raise ValueError("num_bins must be at least 2")
        self.num_bins = int(num_bins)
        self.positive_histogram = torch.zeros(num_bins, dtype=torch.int64)
        self.negative_histogram = torch.zeros(num_bins, dtype=torch.int64)

    @torch.no_grad()
    def update(self, scores, targets, valid_mask=None):
        scores = scores.detach().reshape(-1).cpu()
        targets = targets.detach().reshape(-1).cpu().bool()
        if scores.numel() != targets.numel():
            raise ValueError("scores and targets must contain the same number of values")
        if valid_mask is not None:
            valid_mask = valid_mask.detach().reshape(-1).cpu().bool()
            if valid_mask.numel() != scores.numel():
                raise ValueError("valid_mask must match scores")
            scores = scores[valid_mask]
            targets = targets[valid_mask]
        if not scores.numel():
            return
        if not scores.is_floating_point() or not torch.isfinite(scores).all():
            raise ValueError("scores must contain finite floating-point values")
        if scores.min() < 0 or scores.max() > 1:
            raise ValueError("scores must be normalized to [0, 1]")
        bins = (scores * self.num_bins).long().clamp(max=self.num_bins - 1)
        self.positive_histogram += torch.bincount(bins[targets], minlength=self.num_bins)
        self.negative_histogram += torch.bincount(bins[~targets], minlength=self.num_bins)

    def compute(self):
        positives = int(self.positive_histogram.sum())
        negatives = int(self.negative_histogram.sum())
        if positives == 0 or negatives == 0:
            raise ValueError("AUROC and average precision require both target classes")
        true_positives = self.positive_histogram.flip(0).cumsum(0).double()
        false_positives = self.negative_histogram.flip(0).cumsum(0).double()
        recall = true_positives / positives
        false_positive_rate = false_positives / negatives
        precision = true_positives / (true_positives + false_positives).clamp_min(1)
        zero = torch.zeros(1, dtype=torch.float64)
        auroc = torch.trapezoid(
            torch.cat((zero, recall)), torch.cat((zero, false_positive_rate))
        )
        recall_increment = recall - torch.cat((zero, recall[:-1]))
        average_precision = (precision * recall_increment).sum()
        return {"auroc": float(auroc), "average_precision": float(average_precision)}


class AnomalyMetrics:
    def __init__(self, histogram_bins=2048):
        self.image = ExactBinaryMetrics()
        self.pixel = BinaryHistogramMetrics(histogram_bins)

    @torch.no_grad()
    def update(self, prediction, labels, anomaly_masks, *, valid_pixel_mask=None):
        labels = labels.reshape(-1)
        if labels.shape != prediction.anomaly_score.shape:
            raise ValueError("labels must match anomaly_score shape")
        anomaly_masks = anomaly_masks > 0
        if anomaly_masks.ndim == 3:
            anomaly_masks = anomaly_masks.unsqueeze(1)
        if anomaly_masks.shape != prediction.anomaly_map.shape:
            raise ValueError("anomaly_masks must match anomaly_map shape")
        if valid_pixel_mask is not None:
            valid_pixel_mask = valid_pixel_mask > 0
            if valid_pixel_mask.ndim == 3:
                valid_pixel_mask = valid_pixel_mask.unsqueeze(1)
            if valid_pixel_mask.shape != prediction.anomaly_map.shape:
                raise ValueError("valid_pixel_mask must match anomaly_map shape")
        self.image.update(prediction.anomaly_score, labels)
        self.pixel.update(prediction.anomaly_map, anomaly_masks, valid_pixel_mask)

    def compute(self):
        image = self.image.compute()
        pixel = self.pixel.compute()
        return {
            "image_auroc": image["auroc"],
            "image_average_precision": image["average_precision"],
            "pixel_auroc": pixel["auroc"],
            "pixel_average_precision": pixel["average_precision"],
        }


def update_metrics_from_batch(metrics, prediction, batch):
    valid_pixel_mask = (
        batch["target_mask"] if RESTRICT_PIXELS_TO_TARGET_MASK else None
    )
    metrics.update(
        prediction, batch["label"], batch["anomaly_mask"],
        valid_pixel_mask=valid_pixel_mask,
    )

@torch.no_grad()
def evaluate_anomaly_detector(detector, loader, description, diagnostics=None):
    if not detector.is_fitted:
        raise RuntimeError("The anomaly detector must be fitted before evaluation")
    metrics = AnomalyMetrics(METRIC_HISTOGRAM_BINS)
    detector.train(False)
    for batch in tqdm(loader, desc=description):
        images = batch["image"].to(DEVICE, non_blocking=True)
        if diagnostics is None:
            prediction = detector.predict(images)
            raw_image_scores = None
        else:
            prediction, raw_image_scores, _ = detector.predict_with_raw(images)
        update_metrics_from_batch(metrics, prediction, batch)
        if diagnostics is not None:
            diagnostics.update(prediction, raw_image_scores, batch)
    return metrics.compute()

In [ ]:
def prepare_data(dataset_root):
    # Build preprocessing from the variables declared in the configuration cell.
    evaluation_config = PreprocessingConfig(
        resize=PREPROCESS_RESIZE,
        resize_shorter_side=PREPROCESS_RESIZE_SHORTER_SIDE,
        center_crop=PREPROCESS_CENTER_CROP,
        normalize_mean=NORMALIZE_MEAN,
        normalize_std=NORMALIZE_STD,
        augmentations_enabled=False,
    )
    evaluation_preprocessing = WheelPreprocessor(evaluation_config)
    train_config = PreprocessingConfig(
        resize=PREPROCESS_RESIZE,
        resize_shorter_side=PREPROCESS_RESIZE_SHORTER_SIDE,
        center_crop=PREPROCESS_CENTER_CROP,
        normalize_mean=NORMALIZE_MEAN,
        normalize_std=NORMALIZE_STD,
        augmentations_enabled=TRAIN_AUGMENTATIONS_ENABLED,
        brightness=TRAIN_BRIGHTNESS,
        contrast=TRAIN_CONTRAST,
        gamma=TRAIN_GAMMA,
        saturation=TRAIN_SATURATION,
        sensor_noise=TRAIN_SENSOR_NOISE,
        gaussian_noise=TRAIN_GAUSSIAN_NOISE,
        gaussian_blur=TRAIN_GAUSSIAN_BLUR,
    )
    train_preprocessing = WheelPreprocessor(train_config)

    # Stochastic preprocessing is applied only to the training split.
    train_loader, validation_loader, test_loader = build_dataloaders(
        dataset_root,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        seed=SEED,
        train_preprocessing=train_preprocessing,
        evaluation_preprocessing=evaluation_preprocessing,
    )
    loaders = (
        ("train", train_loader),
        ("validation", validation_loader),
        ("test", test_loader),
    )
    observed_counts = Counter(
        (row["split"], row["condition"])
        for _, loader in loaders
        for row in loader.dataset.rows
    )
    if observed_counts != EXPECTED_SPLIT_CONDITION_COUNTS:
        raise ValueError(
            f"Unexpected split/condition counts: {dict(observed_counts)}; "
            f"expected {dict(EXPECTED_SPLIT_CONDITION_COUNTS)}"
        )
    return {
        "train_config": train_config,
        "train_preprocessing": train_preprocessing,
        "evaluation_config": evaluation_config,
        "evaluation_preprocessing": evaluation_preprocessing,
        "train_loader": train_loader,
        "validation_loader": validation_loader,
        "test_loader": test_loader,
        "loaders": loaders,
    }

In [ ]:
def audit_preprocessing(
    root: str | Path,
    config: PreprocessingConfig,
    *,
    split: str = "train",
    indices: Sequence[int] | None = None,
    variants: int = 2,
    seed: int = 42,
) -> plt.Figure:
    if variants < 1:
        raise ValueError("variants must be at least 1")
    dataset = CuriosityWheelDataset(root, split)
    selected = list(indices) if indices is not None else list(range(min(4, len(dataset))))
    if not selected:
        raise ValueError("indices must select at least one sample")
    if any(index < 0 or index >= len(dataset) for index in selected):
        raise IndexError("audit sample index is outside the dataset")

    preprocessor = WheelPreprocessor(config)
    raw_preprocessor = WheelPreprocessor()
    figure, axes = plt.subplots(
        len(selected), variants + 1,
        figsize=(4.5 * (variants + 1), 3.5 * len(selected)),
        squeeze=False,
    )
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        for row_index, sample_index in enumerate(selected):
            sample = dataset[sample_index]
            raw_image = sample["image"]
            target_mask = sample["target_mask"]
            anomaly_mask = sample["anomaly_mask"]
            image_id = sample["metadata"]["image_id"]
            axes[row_index, 0].imshow(
                raw_preprocessor.image_for_display(raw_image).permute(1, 2, 0)
            )
            axes[row_index, 0].set_title(f"Originale\n{image_id}")
            for variant in range(variants):
                processed, _, _ = preprocessor(
                    raw_image.clone(), target_mask.clone(), anomaly_mask.clone()
                )
                axes[row_index, variant + 1].imshow(
                    preprocessor.image_for_display(processed).permute(1, 2, 0)
                )
                axes[row_index, variant + 1].set_title(f"Augmentata {variant + 1}")
            for axis in axes[row_index]:
                axis.axis("off")
    figure.suptitle(f"Audit preprocessing — split {split}")
    figure.tight_layout()
    return figure

In [ ]:
def smoke_test_loaders(loaders):
    # Verify the common numerical contract on one batch per split.
    for split, loader in loaders:
        batch = next(iter(loader))
        images = batch["image"]
        if images.dtype != torch.float32 or not torch.isfinite(images).all():
            raise TypeError(f"{split} images must be finite float32 tensors")
        if (
            batch["target_mask"].dtype != torch.uint8
            or batch["anomaly_mask"].dtype != torch.uint8
        ):
            raise TypeError(f"{split} masks must remain uint8 tensors")
        print(
            f"{split:10s} image={tuple(images.shape)} {images.dtype} "
            f"range=[{images.min().item():.3f}, {images.max().item():.3f}] "
            f"target_mask={tuple(batch['target_mask'].shape)} "
            f"anomaly_mask={tuple(batch['anomaly_mask'].shape)} "
            f"label={batch['label'].tolist()}"
        )

In [ ]:
@torch.no_grad()
def collect_anomaly_visualization_samples(
    detector, loader, *, device, preprocessor, num_clean=3, num_anomalous=3
):
    if num_clean < 0 or num_anomalous < 0 or num_clean + num_anomalous == 0:
        raise ValueError("At least one non-negative visualization quota is required")
    if not detector.is_fitted:
        raise RuntimeError("The anomaly detector must be fitted before visualization")
    device = torch.device(device)
    detector.to(device)
    detector.train(False)
    remaining = {0: int(num_clean), 1: int(num_anomalous)}
    samples = []
    for batch in loader:
        labels = torch.as_tensor(batch["label"]).reshape(-1)
        selected = [
            index for index, label in enumerate(labels.tolist())
            if label in remaining and remaining[label] > 0
        ]
        if not selected:
            continue
        images = batch["image"][selected].to(device, non_blocking=True)
        prediction = detector.predict(images)
        metadata = batch.get("metadata", {})
        image_ids = metadata.get("image_id", [None] * labels.numel())
        for prediction_index, batch_index in enumerate(selected):
            label = int(labels[batch_index])
            if remaining[label] == 0:
                continue
            samples.append({
                "image": preprocessor.image_for_display(
                    batch["image"][batch_index]
                ).cpu(),
                "target_mask": batch["target_mask"][batch_index].cpu(),
                "anomaly_mask": batch["anomaly_mask"][batch_index].cpu(),
                "anomaly_map": prediction.anomaly_map[prediction_index].cpu(),
                "anomaly_score": float(
                    prediction.anomaly_score[prediction_index].cpu()
                ),
                "label": label,
                "image_id": image_ids[batch_index],
            })
            remaining[label] -= 1
        if not any(remaining.values()):
            break
    if any(remaining.values()):
        raise ValueError(
            "The loader does not contain enough samples for the requested quotas: "
            f"missing clean={remaining[0]}, anomalous={remaining[1]}"
        )
    return samples


def plot_anomaly_visualizations(
    samples, *, title, colormap="magma", overlay_alpha=0.55
):
    if not samples:
        raise ValueError("samples cannot be empty")
    if not 0 <= overlay_alpha <= 1:
        raise ValueError("overlay_alpha must be in [0, 1]")
    figure, axes = plt.subplots(
        len(samples), 4, figsize=(16, 3.7 * len(samples)),
        squeeze=False, constrained_layout=True,
    )
    heatmap_artist = None
    for row, sample in enumerate(samples):
        image = sample["image"].permute(1, 2, 0).numpy()
        target_mask = sample["target_mask"].squeeze().numpy() > 0
        anomaly_mask = sample["anomaly_mask"].squeeze().numpy() > 0
        anomaly_map = sample["anomaly_map"].squeeze().numpy()
        label = "anomalous" if sample["label"] else "clean"
        image_id = sample.get("image_id") or "unknown"
        axes[row, 0].imshow(image)
        axes[row, 0].set_title(
            f"Input — {label}\nscore={sample['anomaly_score']:.3f} | {image_id}"
        )
        axes[row, 1].imshow(anomaly_mask, cmap="gray", vmin=0, vmax=1)
        if target_mask.any() and not target_mask.all():
            axes[row, 1].contour(target_mask, levels=[0.5], colors="cyan", linewidths=1)
        axes[row, 1].set_title("Ground truth\nwhite=hole, cyan=wheel")
        heatmap_artist = axes[row, 2].imshow(
            anomaly_map, cmap=colormap, vmin=0, vmax=1
        )
        if target_mask.any() and not target_mask.all():
            axes[row, 2].contour(target_mask, levels=[0.5], colors="cyan", linewidths=1)
        axes[row, 2].set_title("Anomaly map\nnormalized score [0, 1]")
        axes[row, 3].imshow(image)
        axes[row, 3].imshow(
            anomaly_map, cmap=colormap, vmin=0, vmax=1, alpha=overlay_alpha
        )
        if anomaly_mask.any() and not anomaly_mask.all():
            axes[row, 3].contour(
                anomaly_mask, levels=[0.5], colors="lime", linewidths=1.5
            )
        axes[row, 3].set_title("Overlay\ngreen=ground-truth boundary")
        for axis in axes[row]:
            axis.axis("off")
    figure.suptitle(title, fontsize=15)
    figure.colorbar(
        heatmap_artist, ax=axes[:, 2:].ravel().tolist(),
        label="Normalized anomaly score", shrink=0.8,
    )
    return figure


def save_anomaly_visualizations(
    detector, loader, output_path, *, device, preprocessor, title,
    num_clean=3, num_anomalous=3, colormap="magma",
    overlay_alpha=0.55, dpi=150,
):
    if dpi < 1:
        raise ValueError("dpi must be at least 1")
    samples = collect_anomaly_visualization_samples(
        detector, loader, device=device, preprocessor=preprocessor,
        num_clean=num_clean, num_anomalous=num_anomalous,
    )
    figure = plot_anomaly_visualizations(
        samples, title=title, colormap=colormap, overlay_alpha=overlay_alpha
    )
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_path, dpi=dpi, bbox_inches="tight")
    return figure, output_path

In [ ]:
DEFAULT_GROUP_FIELDS = ("severity", "camera_pose", "lighting", "wear")


def _score_statistics(values: list[float]) -> dict[str, float | int]:
    if not values:
        return {"count": 0}
    scores = torch.tensor(values, dtype=torch.float64)
    return {
        "count": scores.numel(),
        "minimum": float(scores.min()),
        "q25": float(torch.quantile(scores, 0.25)),
        "median": float(torch.quantile(scores, 0.50)),
        "q75": float(torch.quantile(scores, 0.75)),
        "maximum": float(scores.max()),
        "mean": float(scores.mean()),
        "std": float(scores.std(unbiased=False)),
    }


def _connected_components(mask: torch.Tensor) -> list[list[int]]:
    """Return 4-connected positive regions as flattened pixel indices."""
    mask = mask.detach().cpu().bool().squeeze()
    if mask.ndim != 2:
        raise ValueError("PRO masks must be two-dimensional after squeezing")
    height, width = mask.shape
    remaining = set(torch.nonzero(mask.flatten(), as_tuple=False).flatten().tolist())
    components: list[list[int]] = []
    while remaining:
        start = remaining.pop()
        queue = deque([start])
        component = [start]
        while queue:
            index = queue.popleft()
            row, column = divmod(index, width)
            neighbours = []
            if row > 0:
                neighbours.append(index - width)
            if row + 1 < height:
                neighbours.append(index + width)
            if column > 0:
                neighbours.append(index - 1)
            if column + 1 < width:
                neighbours.append(index + 1)
            for neighbour in neighbours:
                if neighbour in remaining:
                    remaining.remove(neighbour)
                    queue.append(neighbour)
                    component.append(neighbour)
        components.append(component)
    return components


class PerRegionOverlap:
    """Memory-bounded PRO curve and normalized AUPRO accumulator."""

    def __init__(self, num_bins: int = 256, max_fpr: float = 0.30) -> None:
        if num_bins < 2:
            raise ValueError("num_bins must be at least 2")
        if not 0 < max_fpr <= 1:
            raise ValueError("max_fpr must be in (0, 1]")
        self.num_bins = int(num_bins)
        self.max_fpr = float(max_fpr)
        self.normal_histogram = torch.zeros(num_bins, dtype=torch.int64)
        self.region_overlap_sum = torch.zeros(num_bins, dtype=torch.float64)
        self.num_regions = 0

    @torch.no_grad()
    def update(
        self,
        scores: torch.Tensor,
        anomaly_masks: torch.Tensor,
        valid_mask: torch.Tensor | None = None,
    ) -> None:
        scores = scores.detach().cpu()
        anomaly_masks = anomaly_masks.detach().cpu() > 0
        if scores.ndim == 3:
            scores = scores.unsqueeze(1)
        if anomaly_masks.ndim == 3:
            anomaly_masks = anomaly_masks.unsqueeze(1)
        if scores.shape != anomaly_masks.shape:
            raise ValueError("scores and anomaly_masks must have matching shapes")
        if valid_mask is None:
            valid_mask = torch.ones_like(anomaly_masks, dtype=torch.bool)
        else:
            valid_mask = valid_mask.detach().cpu() > 0
            if valid_mask.ndim == 3:
                valid_mask = valid_mask.unsqueeze(1)
            if valid_mask.shape != scores.shape:
                raise ValueError("valid_mask must match scores")

        for score, anomaly_mask, valid in zip(scores, anomaly_masks, valid_mask):
            score = score.squeeze(0)
            anomaly_mask = anomaly_mask.squeeze(0) & valid.squeeze(0)
            valid = valid.squeeze(0)
            normal_scores = score[valid & ~anomaly_mask]
            if normal_scores.numel():
                bins = (normal_scores * self.num_bins).long().clamp(
                    min=0, max=self.num_bins - 1
                )
                self.normal_histogram += torch.bincount(
                    bins, minlength=self.num_bins
                )
            flattened_scores = score.flatten()
            for component in _connected_components(anomaly_mask):
                component_scores = flattened_scores[component]
                bins = (component_scores * self.num_bins).long().clamp(
                    min=0, max=self.num_bins - 1
                )
                histogram = torch.bincount(bins, minlength=self.num_bins)
                self.region_overlap_sum += (
                    histogram.flip(0).cumsum(0).double() / len(component)
                )
                self.num_regions += 1

    @staticmethod
    def _normalized_area(
        fpr: torch.Tensor,
        pro: torch.Tensor,
        max_fpr: float,
    ) -> float:
        below = fpr <= max_fpr
        clipped_fpr = fpr[below]
        clipped_pro = pro[below]
        if clipped_fpr[-1] < max_fpr:
            upper_index = int(torch.nonzero(fpr > max_fpr)[0])
            lower_index = upper_index - 1
            denominator = fpr[upper_index] - fpr[lower_index]
            weight = (
                0.0
                if denominator == 0
                else float((max_fpr - fpr[lower_index]) / denominator)
            )
            interpolated = pro[lower_index] + weight * (
                pro[upper_index] - pro[lower_index]
            )
            clipped_fpr = torch.cat(
                (clipped_fpr, torch.tensor([max_fpr], dtype=torch.float64))
            )
            clipped_pro = torch.cat((clipped_pro, interpolated.reshape(1)))
        area = torch.trapezoid(clipped_pro, clipped_fpr) / max_fpr
        return float(area)

    def compute(self) -> dict[str, Any]:
        total_normal = int(self.normal_histogram.sum())
        if total_normal == 0 or self.num_regions == 0:
            raise ValueError("PRO requires normal pixels and at least one anomaly region")
        fpr = self.normal_histogram.flip(0).cumsum(0).double() / total_normal
        pro = self.region_overlap_sum / self.num_regions
        fpr = torch.cat((torch.zeros(1, dtype=torch.float64), fpr))
        pro = torch.cat((torch.zeros(1, dtype=torch.float64), pro))

        reported_max_fprs = sorted({0.05, 0.10, self.max_fpr})
        aupro_by_max_fpr = {
            f"{max_fpr:.2f}": self._normalized_area(fpr, pro, max_fpr)
            for max_fpr in reported_max_fprs
        }
        return {
            "aupro": aupro_by_max_fpr[f"{self.max_fpr:.2f}"],
            "max_fpr": self.max_fpr,
            "aupro_by_max_fpr": aupro_by_max_fpr,
            "num_regions": self.num_regions,
            "curve": [
                {"false_positive_rate": float(x), "pro": float(y)}
                for x, y in zip(fpr, pro)
            ],
        }


class AnomalyDiagnostics:
    """Collect score distributions, grouped image/pixel metrics, and PRO."""

    def __init__(
        self,
        manifest_rows: Sequence[Mapping[str, str]],
        *,
        preprocessor: WheelPreprocessor,
        histogram_bins: int = 2048,
        pro_bins: int = 256,
        pro_max_fpr: float = 0.30,
        group_fields: Sequence[str] = DEFAULT_GROUP_FIELDS,
        num_extreme_examples: int = 6,
        restrict_pixels_to_target_mask: bool = False,
    ) -> None:
        if num_extreme_examples < 1:
            raise ValueError("num_extreme_examples must be at least 1")
        self.preprocessor = preprocessor
        self.histogram_bins = int(histogram_bins)
        self.group_fields = tuple(group_fields)
        self.num_extreme_examples = int(num_extreme_examples)
        self.restrict_pixels_to_target_mask = bool(restrict_pixels_to_target_mask)
        self.pro = PerRegionOverlap(pro_bins, pro_max_fpr)
        self.records: list[dict[str, Any]] = []
        self.group_metrics: dict[str, dict[str, BinaryHistogramMetrics]] = {
            field: {} for field in self.group_fields
        }
        self.group_image_counts: dict[str, dict[str, int]] = {
            field: {} for field in self.group_fields
        }
        self.top_clean: list[dict[str, Any]] = []
        self.bottom_hole: list[dict[str, Any]] = []
        self.anomalous_pixels = 0
        self.valid_pixels = 0
        self.pair_groups = {
            row.get("pair_id", ""): {
                field: row.get(field, "") for field in self.group_fields
            }
            for row in manifest_rows
            if row.get("condition") == "hole" and row.get("pair_id")
        }

    @staticmethod
    def _batch_value(metadata: Mapping[str, Any], key: str, index: int) -> Any:
        values = metadata.get(key)
        if values is None:
            return ""
        value = values[index]
        return value.item() if isinstance(value, torch.Tensor) else value

    def _group_value(
        self, metadata: Mapping[str, Any], field: str, index: int
    ) -> str:
        value = str(self._batch_value(metadata, field, index) or "")
        if value:
            return value
        pair_id = str(self._batch_value(metadata, "pair_id", index) or "")
        return str(self.pair_groups.get(pair_id, {}).get(field, ""))

    def _example(
        self,
        batch: Mapping[str, Any],
        prediction: AnomalyPrediction,
        raw_scores: torch.Tensor,
        index: int,
    ) -> dict[str, Any]:
        metadata = batch.get("metadata", {})
        return {
            "image": self.preprocessor.image_for_display(batch["image"][index]).cpu(),
            "target_mask": batch["target_mask"][index].cpu(),
            "anomaly_mask": batch["anomaly_mask"][index].cpu(),
            "anomaly_map": prediction.anomaly_map[index].detach().cpu(),
            "anomaly_score": float(prediction.anomaly_score[index].detach().cpu()),
            "raw_anomaly_score": float(raw_scores[index].detach().cpu()),
            "label": int(batch["label"][index]),
            "image_id": self._batch_value(metadata, "image_id", index),
        }

    @torch.no_grad()
    def update(
        self,
        prediction: AnomalyPrediction,
        raw_image_scores: torch.Tensor,
        batch: Mapping[str, Any],
    ) -> None:
        raw_image_scores = raw_image_scores.detach().reshape(-1).cpu()
        labels = torch.as_tensor(batch["label"]).reshape(-1).cpu()
        if raw_image_scores.shape != labels.shape:
            raise ValueError("raw_image_scores must match batch labels")
        anomaly_masks = batch["anomaly_mask"] > 0
        valid_mask = (
            batch["target_mask"] > 0
            if self.restrict_pixels_to_target_mask
            else torch.ones_like(anomaly_masks, dtype=torch.bool)
        )
        self.anomalous_pixels += int((anomaly_masks & valid_mask).sum())
        self.valid_pixels += int(valid_mask.sum())
        self.pro.update(prediction.anomaly_map, anomaly_masks, valid_mask)
        metadata = batch.get("metadata", {})

        for index, label_tensor in enumerate(labels):
            label = int(label_tensor)
            condition = "hole" if label else "clean"
            record = {
                "image_id": self._batch_value(metadata, "image_id", index),
                "pair_id": self._batch_value(metadata, "pair_id", index),
                "condition": condition,
                "raw_image_score": float(raw_image_scores[index]),
                "normalized_image_score": float(prediction.anomaly_score[index].cpu()),
            }
            for field in self.group_fields:
                value = self._group_value(metadata, field, index)
                record[field] = value
                if not value:
                    continue
                accumulator = self.group_metrics[field].setdefault(
                    value, BinaryHistogramMetrics(self.histogram_bins)
                )
                accumulator.update(
                    prediction.anomaly_map[index],
                    anomaly_masks[index],
                    valid_mask=valid_mask[index],
                )
                counts = self.group_image_counts[field]
                counts[value] = counts.get(value, 0) + 1
            self.records.append(record)

            example = self._example(batch, prediction, raw_image_scores, index)
            if label == 0:
                self.top_clean.append(example)
                self.top_clean.sort(
                    key=lambda item: item["raw_anomaly_score"], reverse=True
                )
                del self.top_clean[self.num_extreme_examples :]
            else:
                self.bottom_hole.append(example)
                self.bottom_hole.sort(key=lambda item: item["raw_anomaly_score"])
                del self.bottom_hole[self.num_extreme_examples :]

    def compute(self) -> dict[str, Any]:
        if not self.records or self.valid_pixels == 0:
            raise ValueError("Diagnostics require at least one evaluated image")
        grouped: dict[str, dict[str, Any]] = {}
        for field, values in self.group_metrics.items():
            grouped[field] = {}
            for value, metric in sorted(values.items()):
                positives = int(metric.positive_histogram.sum())
                negatives = int(metric.negative_histogram.sum())
                result = metric.compute() if positives and negatives else None
                grouped[field][value] = {
                    "pixel_average_precision": (
                        None if result is None else result["average_precision"]
                    ),
                    "anomalous_pixel_fraction": (
                        positives / (positives + negatives)
                        if positives + negatives
                        else 0.0
                    ),
                    "num_images": self.group_image_counts[field][value],
                }
        image_grouped: dict[str, dict[str, Any]] = {}
        for field in self.group_fields:
            image_grouped[field] = {}
            values = sorted(
                {str(row[field]) for row in self.records if row[field]}
            )
            for value in values:
                records = [row for row in self.records if row[field] == value]
                labels = torch.tensor(
                    [row["condition"] == "hole" for row in records],
                    dtype=torch.bool,
                )
                positives = int(labels.sum())
                negatives = labels.numel() - positives
                metrics = None
                if positives and negatives:
                    accumulator = ExactBinaryMetrics()
                    accumulator.update(
                        torch.tensor(
                            [row["normalized_image_score"] for row in records],
                            dtype=torch.float32,
                        ),
                        labels,
                    )
                    metrics = accumulator.compute()
                image_grouped[field][value] = {
                    "image_auroc": None if metrics is None else metrics["auroc"],
                    "image_average_precision": (
                        None if metrics is None else metrics["average_precision"]
                    ),
                    "num_images": len(records),
                    "num_clean_images": negatives,
                    "num_anomalous_images": positives,
                }

        raw_scores = {
            condition: [
                row["raw_image_score"]
                for row in self.records
                if row["condition"] == condition
            ]
            for condition in ("clean", "hole")
        }
        return {
            "raw_image_score_distribution": {
                condition: _score_statistics(values)
                for condition, values in raw_scores.items()
            },
            "image_metrics_by_group": image_grouped,
            "pixel_average_precision_by_group": grouped,
            "anomalous_pixel_fraction": self.anomalous_pixels / self.valid_pixels,
            "random_pixel_ap_baseline": self.anomalous_pixels / self.valid_pixels,
            "pro": self.pro.compute(),
        }


def save_anomaly_diagnostics(
    diagnostics: AnomalyDiagnostics,
    output_dir: str | Path,
    *,
    split: str,
    colormap: str = "magma",
    overlay_alpha: float = 0.55,
    dpi: int = 150,
) -> tuple[dict[str, Any], list[Path]]:
    """Save diagnostic JSON, score/PRO CSVs, plots, and extreme examples."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    result = diagnostics.compute()
    json_result = {**result, "pro": {k: v for k, v in result["pro"].items() if k != "curve"}}
    summary_path = output_dir / f"{split}_diagnostics.json"
    summary_path.write_text(json.dumps(json_result, indent=2) + "\n", encoding="utf-8")

    score_path = output_dir / f"{split}_image_scores.csv"
    with score_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(diagnostics.records[0]))
        writer.writeheader()
        writer.writerows(diagnostics.records)

    pro_path = output_dir / f"{split}_pro_curve.csv"
    with pro_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["false_positive_rate", "pro"])
        writer.writeheader()
        writer.writerows(result["pro"]["curve"])

    score_figure = Figure(figsize=(8, 5), layout="constrained")
    FigureCanvasAgg(score_figure)
    axis = score_figure.subplots()
    for condition, color in (("clean", "tab:blue"), ("hole", "tab:orange")):
        values = [
            row["raw_image_score"]
            for row in diagnostics.records
            if row["condition"] == condition
        ]
        axis.hist(values, bins=50, density=True, alpha=0.55, label=condition, color=color)
    axis.set_title(f"{split.title()} raw image-score distribution")
    axis.set_xlabel("Raw PatchCore image score")
    axis.set_ylabel("Density")
    axis.legend()
    score_plot_path = output_dir / f"{split}_raw_score_distribution.png"
    score_figure.savefig(score_plot_path, dpi=dpi, bbox_inches="tight")
    score_figure.clear()

    paths = [summary_path, score_path, pro_path, score_plot_path]
    for name, examples, title in (
        ("highest_clean_scores", diagnostics.top_clean, "Clean images with highest raw scores"),
        ("lowest_hole_scores", diagnostics.bottom_hole, "Hole images with lowest raw scores"),
    ):
        if not examples:
            continue
        figure = plot_anomaly_visualizations(
            examples,
            title=f"{split.title()} — {title}",
            colormap=colormap,
            overlay_alpha=overlay_alpha,
        )
        path = output_dir / f"{split}_{name}.png"
        figure.savefig(path, dpi=dpi, bbox_inches="tight")
        figure.clear()
        paths.append(path)
    return json_result, paths


In [ ]:
# Save locally first, then optionally copy the same run artifacts to Drive.
DRIVE_SCOPE = "https://www.googleapis.com/auth/drive.file"
DRIVE_FOLDER_MIME = "application/vnd.google-apps.folder"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def save_json_atomic(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(data, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)
    return path

def get_kaggle_secret(name, required=False):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing required Kaggle secret: {name}")
    return value

def build_drive_service():
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    credentials = Credentials(
        token=None,
        refresh_token=get_kaggle_secret("GDRIVE_REFRESH_TOKEN", required=True),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_kaggle_secret("GDRIVE_CLIENT_ID", required=True),
        client_secret=get_kaggle_secret("GDRIVE_CLIENT_SECRET", required=True),
        scopes=[DRIVE_SCOPE],
    )
    return build("drive", "v3", credentials=credentials, cache_discovery=False)

def drive_children(service, parent_id, name=None):
    escaped_parent = parent_id.replace("'", "\\'")
    query = [f"'{escaped_parent}' in parents", "trashed = false"]
    if name is not None:
        escaped_name = name.replace("\\", "\\\\").replace("'", "\\'")
        query.append(f"name = '{escaped_name}'")
    return service.files().list(
        q=" and ".join(query),
        spaces="drive",
        fields="files(id,name,mimeType,size)",
        pageSize=1000,
    ).execute(num_retries=3).get("files", [])

def ensure_drive_folder(service, parent_id, name, fail_if_exists=False):
    matches = [
        item for item in drive_children(service, parent_id, name)
        if item["mimeType"] == DRIVE_FOLDER_MIME
    ]
    if matches:
        if fail_if_exists:
            raise FileExistsError(f"Drive folder already exists: {name}")
        if len(matches) > 1:
            raise RuntimeError(f"Multiple Drive folders named {name}")
        return matches[0]["id"]
    return service.files().create(
        body={"name": name, "mimeType": DRIVE_FOLDER_MIME, "parents": [parent_id]},
        fields="id",
    ).execute(num_retries=3)["id"]

def drive_upload_file(service, path, parent_id):
    from googleapiclient.http import MediaFileUpload
    path = Path(path)
    media = MediaFileUpload(
        str(path), mimetype="application/octet-stream",
        chunksize=8 * 1024 * 1024, resumable=True,
    )
    request = service.files().create(
        body={"name": path.name, "parents": [parent_id]},
        media_body=media, fields="id,name,size",
    )
    response = None
    while response is None:
        _, response = request.next_chunk(num_retries=5)
    return response

def upload_run_artifacts(paths):
    service = build_drive_service()
    parent = DRIVE_PARENT_FOLDER_ID or get_kaggle_secret("GDRIVE_FOLDER_ID", required=True)
    for folder_name in ("anomaly_detection", MODEL_RUN_NAME):
        parent = ensure_drive_folder(service, parent, folder_name)
    run_folder = ensure_drive_folder(service, parent, RUN_ID, fail_if_exists=True)
    uploaded, failures = [], []
    for path in paths:
        path = Path(path)
        if path.suffix == ".ckpt" and not DRIVE_UPLOAD_CHECKPOINTS:
            continue
        try:
            response = drive_upload_file(service, path, run_folder)
            uploaded.append({
                "name": path.name,
                "drive_file_id": response["id"],
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            })
        except Exception as error:
            failures.append({"name": path.name, "error": type(error).__name__})
    return service, {
        "status": "complete" if not failures else "partial",
        "drive_folder_id": run_folder,
        "uploaded": uploaded,
        "failures": failures,
    }

def persist_run_artifacts(
    detector, validation_results, test_results, additional_artifacts=()
):
    if not detector.is_fitted:
        raise RuntimeError("Cannot save an unfitted anomaly detector")
    for split_results in (validation_results, test_results):
        if set(split_results) != set(METRIC_NAMES):
            raise ValueError(f"Results must contain only: {METRIC_NAMES}")
    results = {"validation": validation_results, "test": test_results}
    additional_artifacts = [Path(path) for path in additional_artifacts]
    if any(not path.is_file() for path in additional_artifacts):
        raise FileNotFoundError("An additional run artifact is missing")
    run_metadata = {
        "run_id": RUN_ID,
        "model_run_name": MODEL_RUN_NAME,
        "seed": SEED,
        "fit_summary": detector.fit_summary,
        "model": {
            "name": MODEL_NAME,
            "pretrained_initialization": getattr(detector, "pretrained", MODEL_NAME == "patchcore" and PATCHCORE_PRETRAINED),
            **detector.checkpoint_config(),
        },
        "preprocessing": {
            "preset": PATCHCORE_PRESET,
            "input_size": list(IMAGE_SIZE),
            "resize": None if PREPROCESS_RESIZE is None else list(PREPROCESS_RESIZE),
            "resize_shorter_side": PREPROCESS_RESIZE_SHORTER_SIDE,
            "center_crop": None if PREPROCESS_CENTER_CROP is None else list(PREPROCESS_CENTER_CROP),
            "normalize_mean": None if NORMALIZE_MEAN is None else list(NORMALIZE_MEAN),
            "normalize_std": None if NORMALIZE_STD is None else list(NORMALIZE_STD),
            "train_augmentations_enabled": TRAIN_AUGMENTATIONS_ENABLED,
            "train_augmentation_parameters": {
                "brightness": TRAIN_BRIGHTNESS,
                "contrast": TRAIN_CONTRAST,
                "gamma": TRAIN_GAMMA,
                "saturation": TRAIN_SATURATION,
                "sensor_noise": TRAIN_SENSOR_NOISE,
                "gaussian_noise": TRAIN_GAUSSIAN_NOISE,
                "gaussian_blur": (
                    None if TRAIN_GAUSSIAN_BLUR is None
                    else list(TRAIN_GAUSSIAN_BLUR)
                ),
            },
        },
        "dataset": {
            "manifest_sha256": sha256_file(DATASET_ROOT / "samples.csv"),
            "expected_split_condition_counts": [
                {"split": split, "condition": condition, "count": count}
                for (split, condition), count
                in sorted(EXPECTED_SPLIT_CONDITION_COUNTS.items())
            ],
        },
        "evaluation": {
            "metrics": list(METRIC_NAMES),
            "pixel_histogram_bins": METRIC_HISTOGRAM_BINS,
            "restrict_pixels_to_target_mask": RESTRICT_PIXELS_TO_TARGET_MASK,
            "diagnostics_enabled": DIAGNOSTICS_ENABLED,
            "diagnostics_group_fields": list(DIAGNOSTICS_GROUP_FIELDS),
            "pro_max_fpr": DIAGNOSTICS_PRO_MAX_FPR,
            "visualization_enabled": VISUALIZATION_ENABLED,
            "visualization_num_clean": VISUALIZATION_NUM_CLEAN,
            "visualization_num_anomalous": VISUALIZATION_NUM_ANOMALOUS,
        },
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    checkpoint_path = detector.save(OUTPUT_DIR / "model.ckpt", metadata=run_metadata)
    metrics_path = save_json_atomic(results, OUTPUT_DIR / "metrics.json")
    manifest_path = save_json_atomic({
        **run_metadata,
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "metrics_sha256": sha256_file(metrics_path),
    }, OUTPUT_DIR / "run_manifest.json")
    artifact_paths = [
        checkpoint_path, metrics_path, manifest_path, *additional_artifacts
    ]
    upload_status = {"status": "disabled"}
    if DRIVE_UPLOAD_ENABLED:
        service, upload_status = upload_run_artifacts(artifact_paths)
        status_path = save_json_atomic(upload_status, OUTPUT_DIR / "upload_status.json")
        drive_upload_file(service, status_path, upload_status["drive_folder_id"])
    else:
        save_json_atomic(upload_status, OUTPUT_DIR / "upload_status.json")
    return {"artifacts": artifact_paths, "upload": upload_status}

## End-to-end execution

Run the remaining cells in order: initialize, prepare and verify data, audit preprocessing, build and fit the model, evaluate, inspect diagnostics, then persist artifacts.

In [ ]:
# Start runtime execution only after every class and function is defined.
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=RESUME)
if MODEL_NAME == "efficientad_s":
    teacher_dir = Path("/kaggle/working/efficientad_assets")
    teacher_dir.mkdir(parents=True, exist_ok=True)
    teacher_path = teacher_dir / "teacher_small.pth"
    if not teacher_path.is_file():
        urllib.request.urlretrieve(EFFICIENTAD_TEACHER_URL, teacher_path)
    SELECTED_MODEL_CONFIG["teacher_weights_path"] = str(teacher_path)
if MODEL_NAME == "tinyglass":
    dtd_archive = Path("/kaggle/working/dtd-r1.0.1.tar.gz")
    dtd_parent = Path("/kaggle/working")
    dtd_images = dtd_parent / "dtd" / "images"
    if not dtd_images.is_dir():
        if not dtd_archive.is_file():
            print("Downloading DTD R1.0.1 for TinyGLASS LAS...")
            urllib.request.urlretrieve(DTD_URL, dtd_archive)
        destination = dtd_parent.resolve()
        with tarfile.open(dtd_archive, "r:gz") as archive:
            for member in archive.getmembers():
                target = (destination / member.name).resolve()
                if destination != target and destination not in target.parents:
                    raise RuntimeError(f"Unsafe DTD archive member: {member.name}")
                if member.issym() or member.islnk():
                    raise RuntimeError(f"DTD archive contains a link: {member.name}")
                if not (member.isfile() or member.isdir()):
                    raise RuntimeError(f"Unsupported DTD archive member: {member.name}")
            archive.extractall(destination)
    dtd_count = sum(
        1 for path in dtd_images.rglob("*")
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    if dtd_count != 5_640:
        raise RuntimeError(
            f"Expected 5,640 DTD R1.0.1 images, found {dtd_count} under {dtd_images}"
        )
    SELECTED_MODEL_CONFIG["texture_root"] = str(dtd_images)
    print(f"TinyGLASS DTD textures: {dtd_count} ({dtd_images})")
DATASET_ROOT = find_dataset_root()
print(f"Dataset: {DATASET_ROOT}")
print(f"Device: {DEVICE}")
print(f"Artifact output directory: {OUTPUT_DIR}")


In [ ]:
data = prepare_data(DATASET_ROOT)
TRAIN_PREPROCESSING_CONFIG = data["train_config"]
TRAIN_PREPROCESSING = data["train_preprocessing"]
EVALUATION_PREPROCESSING_CONFIG = data["evaluation_config"]
EVALUATION_PREPROCESSING = data["evaluation_preprocessing"]
train_loader = data["train_loader"]
validation_loader = data["validation_loader"]
test_loader = data["test_loader"]
loaders = data["loaders"]
for split, loader in loaders:
    print(f"{split:10s}: {len(loader.dataset)} samples, {len(loader)} batches")
print("DataLoaders ready")

In [ ]:
smoke_test_loaders(loaders)
print("Loading verified")

In [ ]:
if AUDIT_ENABLED:
    audit_figure = audit_preprocessing(
        DATASET_ROOT,
        TRAIN_PREPROCESSING_CONFIG,
        split=AUDIT_SPLIT,
        indices=AUDIT_INDICES,
        variants=AUDIT_VARIANTS,
        seed=SEED,
    )
    plt.show()
else:
    print("Preprocessing audit disabled")

In [ ]:
model = build_model()
optimizer = build_optimizer(model)
scheduler = build_scheduler(optimizer)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
print(
    f"Model: {MODEL_NAME} | preset=full | "
    f"parameters={total_parameters:,} | trainable={trainable_parameters:,} | "
    f"optimizer={OPTIMIZER_NAME} | scheduler={SCHEDULER_NAME}"
)

In [ ]:
penalty_loader = None
if MODEL_NAME == "efficientad_s":
    print("[EfficientAD] Reading HF_TOKEN from Kaggle Secrets...", flush=True)
    hf_token = get_kaggle_secret("HF_TOKEN", required=True)
    print(
        "[EfficientAD] HF_TOKEN loaded; preparing the local ImageNet penalty cache.",
        flush=True,
    )
    penalty_loader = build_imagenet_penalty_loader(hf_token)
    print("[EfficientAD] Local ImageNet penalty loader ready.", flush=True)
else:
    print("ImageNet penalty cache not required for the selected model.")


In [ ]:
print(f"[1/6] Fitting {MODEL_RUN_NAME} on {DEVICE}")
fit_kwargs = {"device": DEVICE, "work_dir": OUTPUT_DIR, "resume": RESUME}
if MODEL_NAME != "patchcore":
    fit_kwargs["validation_loader"] = validation_loader
if MODEL_NAME == "efficientad_s":
    if penalty_loader is None:
        raise RuntimeError("Run the ImageNet penalty cache cell before fit")
    fit_kwargs["penalty_loader"] = penalty_loader
print(f"[fit] Entering {type(model).__name__}.fit()", flush=True)
model.fit(train_loader, **fit_kwargs)
print(f"[fit] {type(model).__name__}.fit() returned successfully", flush=True)
print(json.dumps(model.fit_summary, indent=2))
model.save(
    OUTPUT_DIR / "model.ckpt",
    metadata={"run_id": RUN_ID, "fit_summary": model.fit_summary},
)
print("Post-fit checkpoint saved")


In [ ]:
validation_diagnostics = (
    AnomalyDiagnostics(
        validation_loader.dataset.rows,
        preprocessor=EVALUATION_PREPROCESSING,
        histogram_bins=METRIC_HISTOGRAM_BINS,
        pro_bins=DIAGNOSTICS_PRO_BINS,
        pro_max_fpr=DIAGNOSTICS_PRO_MAX_FPR,
        group_fields=DIAGNOSTICS_GROUP_FIELDS,
        num_extreme_examples=DIAGNOSTICS_NUM_EXTREME_EXAMPLES,
        restrict_pixels_to_target_mask=RESTRICT_PIXELS_TO_TARGET_MASK,
    )
    if DIAGNOSTICS_ENABLED else None
)
print("[2/6] Evaluating validation split")
validation_metrics = evaluate_anomaly_detector(
    model, validation_loader, "Validation", diagnostics=validation_diagnostics
)
print(validation_metrics)

In [ ]:
test_diagnostics = (
    AnomalyDiagnostics(
        test_loader.dataset.rows,
        preprocessor=EVALUATION_PREPROCESSING,
        histogram_bins=METRIC_HISTOGRAM_BINS,
        pro_bins=DIAGNOSTICS_PRO_BINS,
        pro_max_fpr=DIAGNOSTICS_PRO_MAX_FPR,
        group_fields=DIAGNOSTICS_GROUP_FIELDS,
        num_extreme_examples=DIAGNOSTICS_NUM_EXTREME_EXAMPLES,
        restrict_pixels_to_target_mask=RESTRICT_PIXELS_TO_TARGET_MASK,
    )
    if DIAGNOSTICS_ENABLED else None
)
print("[3/6] Evaluating test split")
test_metrics = evaluate_anomaly_detector(
    model, test_loader, "Test", diagnostics=test_diagnostics
)
print(test_metrics)

In [ ]:
diagnostics_results = {}
diagnostics_paths = []
if DIAGNOSTICS_ENABLED:
    print("[4/6] Saving score, failure-case, grouped AP, and PRO diagnostics")
    for split, split_diagnostics in (
        ("validation", validation_diagnostics),
        ("test", test_diagnostics),
    ):
        diagnostic_result, split_paths = save_anomaly_diagnostics(
            split_diagnostics,
            OUTPUT_DIR,
            split=split,
            colormap=VISUALIZATION_COLORMAP,
            overlay_alpha=VISUALIZATION_OVERLAY_ALPHA,
            dpi=VISUALIZATION_DPI,
        )
        diagnostics_results[split] = diagnostic_result
        diagnostics_paths.extend(split_paths)
        for path in split_paths:
            if path.suffix.lower() == ".png":
                with Image.open(path) as preview:
                    display(preview.copy())
    print(json.dumps(diagnostics_results, indent=2))
else:
    print("Extended diagnostics disabled")


In [ ]:
visualization_paths = []
if VISUALIZATION_ENABLED:
    print("[5/6] Creating qualitative anomaly visualizations")
    for split, loader in (("validation", validation_loader), ("test", test_loader)):
        figure, visualization_path = save_anomaly_visualizations(
            model,
            loader,
            OUTPUT_DIR / f"{split}_examples.png",
            device=DEVICE,
            preprocessor=EVALUATION_PREPROCESSING,
            title=f"{split.title()} anomaly examples",
            num_clean=VISUALIZATION_NUM_CLEAN,
            num_anomalous=VISUALIZATION_NUM_ANOMALOUS,
            colormap=VISUALIZATION_COLORMAP,
            overlay_alpha=VISUALIZATION_OVERLAY_ALPHA,
            dpi=VISUALIZATION_DPI,
        )
        visualization_paths.append(visualization_path)
        plt.show()
        plt.close(figure)
else:
    print("Qualitative visualization disabled")

In [ ]:
artifact_status = persist_run_artifacts(
    model, validation_metrics, test_metrics,
    additional_artifacts=[*visualization_paths, *diagnostics_paths],
)
print(f"[6/6] Artifacts saved to {OUTPUT_DIR}")
print(artifact_status["upload"])